# Street View Insights (`SV_PANO` + Minimal 3D LiDAR): Decoupled Address-to-House Detection & Front-Door Localization

[Open in Colab](https://colab.research.google.com/github/googlemaps-samples/insights-samples/blob/svi-gemini-improvements/street_view_insights/panoramic/notebooks/address_to_house_triangulation_with_gmp_and_lidar_v2.ipynb) · [Open in Colab Enterprise](https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fsvi-gemini-improvements%2Fstreet_view_insights%2Fpanoramic%2Fnotebooks%2Faddress_to_house_triangulation_with_gmp_and_lidar_v2.ipynb) · [View on GitHub](https://github.com/googlemaps-samples/insights-samples/blob/svi-gemini-improvements/street_view_insights/panoramic/notebooks/address_to_house_triangulation_with_gmp_and_lidar_v2.ipynb)

**Expected runtime:** ~10–15 s per address interactively (~$0.06/address in online `gemini-4-hydrogen` tokens or ~$0.03/address via Vertex AI Batch Prediction; a single batched Float-BBox + `UNNEST(@batch_pts)` BigQuery scan amortizes table-scan cost across hundreds or thousands of route stops).

Given one or more **US delivery addresses**, this notebook combines **Google Maps Platform Geocoding API v4 (GA)** with **Street View Insights panoramic imagery (`SV_PANO`)** and an **on-demand, surgical 3D LiDAR wall-verification fallback** in a **Decoupled 2-Use-Case Architecture** that separately scores and triangulates:

1. **Use Case 1 — Address $\rightarrow$ House Detection (`house_zone`: `GREEN` / `YELLOW` / `RED`)**: Confirm the right physical building from `>= 3` street-facing car viewpoints using **all** target-building rays (`door`, `porch`, `facade_center`), Cauchy RANSAC consensus, and unprimed house-number OCR — **without penalizing a clearly visible house when its front door is hidden behind a tree or parked car**.
2. **Use Case 2 — Conditional Front-Door Localization (`door_zone`: `GREEN` / `YELLOW` / `OCCLUDED` / `SKIPPED`)**: Run **strictly after** a house is detected (`house_zone != RED`), using **pure `door` rays only** (zero `facade_center` mixing) so front-door coordinates stay tight (`~13 cm` median multi-ray RMS, `1.66 m` pure-SVI median entrance error) and cleanly report `OCCLUDED` when foliage or vehicles hide the door.

| Stage | What happens | Services used |
| :--- | :--- | :--- |
| **1. Parcel, Curbside & Entity Gate** | Resolve building outline (`displayPolygon` with `<= 2.0 m` fallback containment check), curbside `navigationPoints`, and typed `subDestinations` (`types`, `structureType`); gate non-physical (`PO Box`/`RR`/`HC`), unresolved explicit-unit (`UNRESOLVED_UNIT_DESTINATION`), street-mismatched (`ENTITY_MISMATCH`), or campus (`GROUNDS` / `> 1,500 m²`, `TARGET_IS_COMPLEX`) inputs before imagery spend | Geocoding API v4 GA (`/v4/geocode/destinations` + `/v4/geocode/address`) |
| **2. Ephemeral Float-BBox Batch Fetch & SVI-Track Selector** | Pass in-memory coordinates via `UNNEST(@batch_pts)` with Float-BBox predicate pushdown (`1` BigQuery scan for `N` stops, `0` persistent GMP tables), orient facades along exact wall tangents (snapping missing or inside-building `navigationPoints` to the nearest `SV_PANO` car trajectory), and pick up to 6 diverse viewpoints | BigQuery (`pano_observations_latest`), fisheye camera geometry |
| **3. Unprimed Zoom-Crop Detection** | Fast `PIL.Image.draft` 1/2-DCT decode + zoom crop (`<= 1280 px`) around the projected building window for unprimed house-number OCR (`ocr_mounted_on`: target vs. neighbor), door/porch detection, privacy-blur opt-out detection, and permitted Logistics & Delivery access attributes | Gemini (`gemini-4-hydrogen`, `thinkingLevel="LOW"`) on Vertex AI, Cloud Storage (in-memory) |
| **4. Decoupled Cauchy RANSAC Multi-Ray Solvers** | Map `0..1000` boxes losslessly back to full-frame sensor pixels, run loose 3D elevation sanity, then run **Stage A (House Detection)** over all building rays (`persistable_house=True`) and **Stage B (Conditional Door Lock)** over pure `door` rays (`persistable_door=True`) with Cauchy RANSAC (`0.85 m` kernel) + post-IRLS outlier trimming (`<= 1.60 m`) + Huber-weighted RMS | NumPy / Shapely |
| **5. Surgical 3D LiDAR Verification** | **Triggered only when Stage-A RGB falls short of `TIER_A`** and `>= 2` views have `>= 12°` parallax: join `pano_lidar_metadata_latest`, fit the farthest planar vertical 3D wall slab (`<= 25°` normal check), and verify `<= 1.0 m` mutual ray agreement | BigQuery (`pano_lidar_metadata_latest`), `laspy` (in-memory) |

### Decoupled Output Tiers & Verified Benchmark (`N = 100` US Addresses, `balanced` Default)

* **Use Case 1 — House Detection (`house_zone`, `N = 100`)**:
  * **`GREEN` (`81%`, `93.8%` main-building precision, `0.00 m` median wall distance)**: Confirmed `TIER_A` building (`>= 3` car positions, `<= 1.65 m` Cauchy/Huber RMS, validated building outline, zero unresolved OCR conflict).
  * **`YELLOW` (`12%`)**: Multi-unit shared structure without unit OCR, `< 3` converging views, missing `displayPolygon`, or converged `TIER_A` house with neighbor OCR discrepancy.
  * **`RED` (`7%`)**: Stage-1 gated inputs (`ENTITY_MISMATCH`, `TARGET_IS_COMPLEX`), homeowner Street View opt-out (`PRIVACY_BLURRED_IN_SVI`), unconverged OCR conflict, or `0` usable views.
* **Use Case 2 — Conditional Front-Door Localization (`door_zone`, given House Detected `N = 93`)**:
  * **`GREEN` (`55.9%` of detected houses, `94.2%` precision, `13 cm` median ray RMS, `1.66 m` pure-SVI held-out entrance median error / `1.12 m` ephemeral UI wall-clamp)**: `DOOR_LOCKED_3PLUS` (`>= 3` pure `door` rays, `<= 1.25 m` RMS, `>= 12°` parallax).
  * **`YELLOW` (`22.6%`)**: `DOOR_LOCKED_2RAY` (`2` pure door rays) or `PORCH_FALLBACK` (`>= 2` porch rays when door is recessed).
  * **`OCCLUDED` (`21.5%`)**: `DOOR_SINGLE_VIEW` (1 visible door frame — ephemeral hint only, `door_lat=None`) or `DOOR_OCCLUDED` (front door hidden behind trees/cars; **House pin remains `GREEN`**!).
  * **`SKIPPED`**: Stage B skipped because Use Case 1 returned `house_zone == RED`.

## 1. Before you start

**Enable these APIs** in your Google Cloud project:
* **BigQuery API**, **Vertex AI API**, and **Cloud Storage API** (for Street View Insights views, imagery streaming, and Gemini inference)
* **Geocoding API** (`geocoding-backend.googleapis.com`) — uses GA v4 `SearchDestinations` and `GeocodeAddress`

**Required IAM roles & Production Authentication:**

| Role / Credential | Purpose |
| :--- | :--- |
| `roles/bigquery.jobUser` & `roles/bigquery.dataViewer` | Run parameterized queries against `imagery_insights___us` (and optional `imagery_insights_lidar___us`) |
| `roles/storage.objectViewer` | Stream Street View Insights JPEG frames and `.laz` sweeps in memory |
| `roles/aiplatform.user` | Call `gemini-4-hydrogen` on Vertex AI (never use the Google AI Studio free tier with GMP/SVI data) |
| `roles/serviceusage.serviceUsageConsumer` or `X-Goog-Api-Key` | Attribute Geocoding API v4 calls. *Note:* Colab development uses `cloud-platform` ADC for convenience; in production services, authenticate Geocoding v4 with an API-restricted `X-Goog-Api-Key` or narrow OAuth scope (`https://www.googleapis.com/auth/maps-platform.geocode`). |

**Terms of Service, Permitted Use Cases, Privacy & Caching Rules:**
* Usage is governed by the [Google Maps Platform Terms of Service](https://cloud.google.com/maps-platform/terms), the [Google Maps Core Services Specific Terms](https://cloud.google.com/terms/maps-platform/maps-service-terms), and the [Google Maps Analytics Service Specific Terms](https://cloud.google.com/terms/maps-platform/maps-analytics-service-terms).
* **Permitted vs. Prohibited Use Cases**: This notebook is scoped strictly to **Logistics & Delivery ("Last 100 Feet")** access attributes (`has_porch`, `has_fence_or_gate`, `driveway_or_staging_visible`, `step_or_curb_access`, `access_notes`). Street View Insights **strictly prohibits** Personal Residential Insurance Underwriting, Consumer Insurance Claim Validation, and Automated Rights-Impacting Decisions.
* **Residential Delivery Privacy Notice (Analytics SST §3.5.1)**: Analyzing private residential properties for delivery workflows requires clear disclosure in your public privacy policy and Terms of Service, consent where locally required, and `>= 10`-residence aggregation for any broad regional scoring.
* **Strict Caching & Persistence Boundaries (GMP ToS §3.2.3 & Analytics SST §3.6 / §3.8)**:
  * **`place_id`**: May be stored **indefinitely**.
  * **Geocoding `(lat, lng)`**: May be cached for at most **30 consecutive calendar days**.
  * **`displayPolygon`, `navigationPoints`, `entrances`, `subDestinations`, `formattedAddress`, `building_centroid_ephemeral_lat/lng`, and `door_display_ephemeral_lat/lng`**: Have a **0-day caching exemption** — keep strictly in ephemeral worker RAM (pass coordinates to BigQuery via in-memory `UNNEST(@batch_pts)` parameters; never persist to BigQuery tables or disk).
  * **Pure SVI Customer Detections & 35-Day Snapshot TTL**: Pure SVI/LiDAR-triangulated `house_lat/lng` (`persistable_house=True`), `door_lat/lng` (`persistable_door=True`), and permitted delivery attributes are Customer Detections, while `observation_id` references expire with SVI's **35-day snapshot TTL** (`<= 24h` on any ephemeral image cache; stream JPEGs/`.laz` in memory).
* **AI / ML Restrictions (GMP ToS §3.2.3(c)(vii) & Analytics SST §3.4)**: Never use Google Maps Geocoding outputs as labels to train, test, validate, or fine-tune ML models. Custom models trained solely on SVI imagery must remain hosted exclusively in GCP/GEAP and retrain at least every 2 years under Analytics SST §3.4.
* **Attribution & Non-Google Basemaps**: Always display **"Google Maps | Imagery © Google"** on local plots and **"Imagery © Google"** on Street View crops. **Never** overlay `displayPolygon` or `navigationPoints` on non-Google basemaps (OpenStreetMap, Mapbox, Folium).
* **Launch Stages**: Geocoding API v4 (`/v4/geocode/destinations` and `/v4/geocode/address`) is **Generally Available (GA)**. Street View Insights `SV_PANO` is **Preview**, and the SVI 3D LiDAR Add-On (`imagery_insights_lidar___us.pano_lidar_metadata_latest`) is **Restricted Experimental** (30-customer allowlist cap) under the [Google Cloud Launch Stages](https://cloud.google.com/terms/launch-stages).

In [ ]:
# @title Step 1a: Install packages
%pip install -q --upgrade google-genai google-cloud-bigquery db-dtypes shapely "laspy[lazrs]"

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# @title Step 1b: Imports & authentication
import concurrent.futures as cf
import dataclasses
from dataclasses import dataclass
import io
import json
import logging
import math
import os
import random
import re
import sys
import threading
import time
import urllib.parse
import warnings
from typing import Any, Mapping, Sequence

from google.api_core.exceptions import GoogleAPICallError
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google.cloud import bigquery
from google import genai
from google.genai import types
import laspy
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from pydantic import BaseModel, Field, ValidationError
from shapely.geometry import LineString, MultiPolygon, Point, Polygon, shape
from shapely.ops import nearest_points

warnings.filterwarnings("ignore", category=UserWarning)
logger = logging.getLogger("svi_house_v2")

if "google.colab" in sys.modules:
    from google.colab import auth as colab_auth
    colab_auth.authenticate_user()

print("Libraries imported and ready.")

## 2. Configure your project and target US address

* `PROJECT_ID`: Google Cloud project that bills BigQuery, Vertex AI, and Geocoding API v4 calls.
* `SVI_PROJECT_ID` / `SVI_DATASET`: your subscribed Street View Insights panoramic dataset (`imagery_insights___us`).
* `SVI_LIDAR_DATASET` & `ENABLE_SURGICAL_LIDAR_FALLBACK`: optional `SV_PANO` 3D LiDAR dataset (`imagery_insights_lidar___us.pano_lidar_metadata_latest`), invoked **only** when RGB falls short of `Tier A` and at least `2` views have `>= 12°` parallax.
* `GMP_MAPS_API_KEY`: optional restricted Geocoding API key (falls back to `GMP_MAPS_API_KEY` env var or OAuth `AuthorizedSession` in Colab).
* `CONFIDENCE_OPERATING_POINT`: `'strict'`, `'balanced'` (default), or `'high_recall'` threshold preset for `GREEN` / `YELLOW` / `RED` classification.

In [ ]:
# @title Step 2: Configuration
PROJECT_ID = "your-gcp-project-id"  # @param {type:"string"}
SVI_PROJECT_ID = ""  # @param {type:"string"}
SVI_DATASET = "imagery_insights___us"  # @param {type:"string"}
SVI_LIDAR_DATASET = "imagery_insights_lidar___us"  # @param {type:"string"}
GMP_MAPS_API_KEY = ""  # @param {type:"string"}
VERTEX_LOCATION = "global"  # @param {type:"string"}
GEMINI_MODEL = "gemini-4-hydrogen"  # @param {type:"string"}
GEMINI_THINKING_LEVEL = "LOW"  # @param ["LOW", "MEDIUM", "HIGH"]
GEMINI_CROP_MAX_DIM = 1280  # @param {type:"integer"}

TARGET_ADDRESS = "530 W Myrtle St, Lakeland, FL 33815"  # @param {type:"string"}
MAX_PANOS = 6  # @param {type:"integer"}
ENABLE_SURGICAL_LIDAR_FALLBACK = True  # @param {type:"boolean"}
CONFIDENCE_OPERATING_POINT = "balanced"  # @param ["strict", "balanced", "high_recall"]
MAX_QUERY_GB = 10.0  # @param {type:"number"}

GEMINI_ONLINE_INPUT_USD_PER_M = 1.50  # @param {type:"number"}
GEMINI_ONLINE_OUTPUT_USD_PER_M = 7.50  # @param {type:"number"}
GEMINI_BATCH_INPUT_USD_PER_M = 0.75  # @param {type:"number"}
GEMINI_BATCH_OUTPUT_USD_PER_M = 3.75  # @param {type:"number"}

if PROJECT_ID == "your-gcp-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
SVI_PROJECT_ID = SVI_PROJECT_ID or os.environ.get("SVI_PROJECT_ID", "") or PROJECT_ID
GMP_MAPS_API_KEY = GMP_MAPS_API_KEY or os.environ.get("GMP_MAPS_API_KEY", "")
assert PROJECT_ID != "your-gcp-project-id", "Set PROJECT_ID to your Google Cloud project ID."

bq_client = bigquery.Client(project=PROJECT_ID)
creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
authed_http = AuthorizedSession(creds)
genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=VERTEX_LOCATION)

print(f"Billing project: {PROJECT_ID} | SVI dataset: {SVI_PROJECT_ID}.{SVI_DATASET}")
print(f"Model: {GEMINI_MODEL} (thinking={GEMINI_THINKING_LEVEL}, max_dim={GEMINI_CROP_MAX_DIM}px)")
print(f"Target address:  {TARGET_ADDRESS} | Operating point: {CONFIDENCE_OPERATING_POINT}")

## 3. Stage 1: Resolve parcel outline, curbside prior & Stage-1 gates via GA Geocoding API v4

A plain rooftop geocode `(lat, lng)` fails on corner lots, divided roads, non-physical postal boxes, and multi-building apartment complexes.

`resolve_address_gmp()` calls **GA Geocoding API v4** (`POST https://geocode.googleapis.com/v4/geocode/destinations`, with `GET /v4/geocode/address` Essentials fallback) and enforces four production guardrails before touching BigQuery or Gemini:
1. **Entity & Non-Physical Gate (`ENTITY_MISMATCH`)**: Rejects `PO Box` / `RR` / `HC` inputs, `APPROXIMATE` locality fallbacks, and street-name/suffix/number mismatches upfront (`$0` imagery spend).
2. **Validated Building Outline (`displayPolygon`)**: Uses `primary.displayPolygon` or `containingPlaces` (`BUILDING`), and accepts a `locationQuery` (`structureType: BUILDING`) fallback **only** when `primary.location` lies within `<= 2.0 m` of the polygon (preventing adjacent-neighbor footprint poisoning).
3. **Campus / Complex Gate (`TARGET_IS_COMPLEX`) & Typed `subDestinations`**: Requests `subDestinations.types` and `subDestinations.structureType` so genuine `subpremise`/`premise` units set `is_multi_unit=True` while home-business POIs do not, and rejects `GROUNDS` or `> 1,500 m²` complexes.
4. **Exact Wall-Tangent Geometry & Immutable `ParcelPrior`**: Computes `right_u` strictly from the front-wall segment tangent `(p1 - p0) / seg_len`, keeps `ParcelPrior(frozen=True)` 100% immutable (`dataclasses.replace`), and flags inside-polygon (`< 2.5 m`), distant (`> 30 m`), or missing `navigationPoints` so Stage 2 snaps street orientation directly to the nearest `SV_PANO` car trajectory.

In [ ]:
# @title Step 3: GA Geocoding v4 resolver, entity/complex gates & pure ParcelPrior
WGS84_A = 6_378_137.0
WGS84_E2 = 0.00669437999014
CAM_HEIGHT_ABOVE_GROUND_M = 2.5

MIN_RAY_RANGE_M = 2.5
MAX_RAY_RANGE_M = 55.0
MIN_TIER_A_PARALLAX_DEG = 10.0
MAX_TIER_A_RMS_M = 1.65

GEOCODE_DESTINATIONS_URL = "https://geocode.googleapis.com/v4/geocode/destinations"
GEOCODE_ADDRESS_URL = "https://geocode.googleapis.com/v4/geocode/address"
DESTINATIONS_FIELDMASK = ",".join([
    "destinations.primary",
    "destinations.containingPlaces",
    "destinations.subDestinations.formattedAddress",
    "destinations.subDestinations.types",
    "destinations.subDestinations.structureType",
    "destinations.navigationPoints",
])

_NON_PHYSICAL_ADDR_RE = re.compile(
    r"^\s*(?:p\.?\s*o\.?\s*box|post\s+office\s+box|rr\s+\d+|hc\s+\d+|rural\s+route|highway\s+contract|general\s+delivery)\b",
    re.IGNORECASE,
)
_EXPLICIT_UNIT_RE = re.compile(
    r"(?:#\s*[A-Za-z0-9-]+|\b(?:apt|apartment|unit|ste|suite|lot|space|bldg|building|rm|room|floor)\b\.?\s*[A-Za-z0-9-]+)",
    re.IGNORECASE,
)
_SUFFIX_NORM = {
    "street": "st", "st": "st", "avenue": "ave", "ave": "ave", "av": "ave",
    "road": "rd", "rd": "rd", "drive": "dr", "dr": "dr", "boulevard": "blvd", "blvd": "blvd",
    "lane": "ln", "ln": "ln", "court": "ct", "ct": "ct", "place": "pl", "pl": "pl",
    "circle": "cir", "cir": "cir", "terrace": "ter", "ter": "ter", "parkway": "pkwy", "pkwy": "pkwy",
    "highway": "hwy", "hwy": "hwy", "way": "way", "trail": "trl", "trl": "trl",
    "north": "n", "south": "s", "east": "e", "west": "w",
    "northeast": "ne", "northwest": "nw", "southeast": "se", "southwest": "sw",
    "county": "co", "co": "co", "us": "us", "state": "sr",
}
_DIRS = {"n", "s", "e", "w", "ne", "nw", "se", "sw"}
_ST_TYPES = {"st", "ave", "rd", "dr", "blvd", "ln", "ct", "pl", "cir", "ter", "pkwy", "way", "trl", "hwy"}
_UNIT_STOP = {"apt", "apartment", "unit", "ste", "suite", "bldg", "building", "fl", "floor", "lot", "space", "spc", "rm", "room"}
_UNIT_SUBDEST_TYPES = {"subpremise", "premise", "room", "floor"}
_COMPLEX_GROUNDS_TYPES = {"apartment_complex", "housing_complex", "mobile_home_park", "condominium_complex", "shopping_mall"}


def _wgs84_radii(lat_deg: float) -> tuple[float, float]:
    lat_r = math.radians(lat_deg)
    sin_l, cos_l = math.sin(lat_r), math.cos(lat_r)
    denom = math.sqrt(1.0 - WGS84_E2 * sin_l * sin_l)
    return WGS84_A * (1.0 - WGS84_E2) / (denom ** 3), (WGS84_A / denom) * cos_l


def lla_to_en(lat: float, lng: float, ref_lat: float, ref_lng: float) -> tuple[float, float]:
    """WGS84 ellipsoidal local East-North meters (aligned with `ecef_to_enu` to <1 mm at 50 m)."""
    m_rad, n_cos = _wgs84_radii(0.5 * (lat + ref_lat))
    return math.radians(lng - ref_lng) * n_cos, math.radians(lat - ref_lat) * m_rad


def en_to_lla(east: float, north: float, ref_lat: float, ref_lng: float) -> tuple[float, float]:
    m_rad0, _ = _wgs84_radii(ref_lat)
    lat = ref_lat + math.degrees(north / m_rad0)
    _, n_cos = _wgs84_radii(0.5 * (lat + ref_lat))
    return lat, ref_lng + math.degrees(east / max(n_cos, 1e-6))


def distance_m(lat1: float, lng1: float, lat2: float, lng2: float) -> float:
    e, n = lla_to_en(lat2, lng2, lat1, lng1)
    return math.hypot(e, n)


def ang_diff_deg(a: float, b: float) -> float:
    return (a - b + 180.0) % 360.0 - 180.0


def angle_between_rays_deg(az1: float, az2: float) -> float:
    d = abs(ang_diff_deg(az1, az2))
    return min(d, 180.0 - d)


def extract_street_number(address: str | None) -> str | None:
    if not address:
        return None
    m = re.match(r"^\s*([NSEW]?\d+(?:-\d+)?[A-Za-z]?)\b", address.strip(), re.IGNORECASE)
    return m.group(1).upper() if m else None


def _parse_street_tokens(raw_segment: str) -> tuple[list[str], str | None]:
    seg = raw_segment.split(",")[0].strip()
    seg = re.sub(r"^\s*(?:[NSEW]\d+\s+|\d+(?:-\d+)?[A-Za-z]?\s+)", "", seg, flags=re.IGNORECASE)
    seg = re.split(
        r"\s+(?:#|(?:apt|apartment|unit|ste|suite|bldg|building|lot|space|rm|room|fl|floor)\b)",
        seg, maxsplit=1, flags=re.IGNORECASE,
    )[0]
    toks = [_SUFFIX_NORM.get(t, t) for t in re.findall(r"[A-Za-z0-9]+", seg.lower()) if t not in _UNIT_STOP]
    sfx = next((t for t in reversed(toks) if t in _ST_TYPES), None)
    return toks, sfx


def _validate_entity_match(
    query_address: str, prim: Mapping[str, Any], route_long_or_short: str | None
) -> tuple[bool, str | None]:
    if _NON_PHYSICAL_ADDR_RE.match(query_address or ""):
        return False, "NON_PHYSICAL_POSTAL_BOX_OR_RURAL_ROUTE"
    if not prim or not prim.get("location"):
        return False, "UNRESOLVED_DESTINATION"

    q_num = extract_street_number(query_address)
    if q_num and str(prim.get("granularity") or "") == "APPROXIMATE":
        return False, "APPROXIMATE_LOCALITY_FALLBACK"

    fmt = str(prim.get("formattedAddress") or "")
    if q_num and fmt:
        q_lead = re.match(r"^([NSEW]?\d+)", q_num, re.IGNORECASE)
        if q_lead:
            q_base = q_lead.group(1).upper()
            fmt_nums = {m.upper() for m in re.findall(r"\b([NSEW]?\d+)\b", fmt, re.IGNORECASE)}
            if q_base not in fmt_nums:
                return False, f"STREET_NUMBER_MISMATCH({q_base}_NOT_IN_RESOLVED)"

    q_toks, q_sfx = _parse_street_tokens(query_address)
    if route_long_or_short:
        r_toks, r_sfx = _parse_street_tokens(route_long_or_short)
    else:
        segs = [s.strip() for s in fmt.split(",")]
        num_seg = next((s for s in segs if re.match(r"^(?:[NSEW]?\d+)", s, re.IGNORECASE)), segs[0] if segs else "")
        r_toks, r_sfx = _parse_street_tokens(num_seg)

    q_core = [t for t in q_toks if t not in _DIRS and t not in _ST_TYPES]
    r_core = [t for t in r_toks if t not in _DIRS and t not in _ST_TYPES]
    if q_core and r_core and not (set(q_core) & set(r_core)):
        return False, f"STREET_NAME_MISMATCH({'_'.join(q_core)}_VS_{'_'.join(r_core)})"
    if q_sfx and r_sfx and q_sfx != r_sfx and q_core == r_core:
        return False, f"STREET_SUFFIX_MISMATCH({q_sfx}_VS_{r_sfx})"
    return True, None


@dataclass(frozen=True)
class ParcelPrior:
    """Immutable parcel, street-reference, and facade prior (never mutated in place)."""

    query_address: str
    formatted_address: str
    place_id: str | None
    street_number: str | None
    route_name: str | None
    access_street_name: str | None
    ref_lat: float
    ref_lng: float
    nav_lat: float
    nav_lng: float
    has_gmp_nav: bool
    nav_outside_poly_m: float
    needs_svi_street_ref: bool
    street_ref_source: str  # "navigationPoint", "svi_car_trajectory", "geocode_fallback"
    is_multi_unit: bool
    unit_sub_destination_count: int
    total_sub_destination_count: int
    polygon_source: str  # "primary", "containingPlaces", "locationQuery_BUILDING", "none"
    polygon_area_m2: float
    dropped_uncontained_fallback_poly: bool
    poly_en: Polygon | None
    front_left_en: np.ndarray
    front_mid_en: np.ndarray
    front_right_en: np.ndarray
    inward_normal_en: np.ndarray
    true_half_depth_m: float
    facade_width_m: float
    stage1_status: str  # "OK", "ENTITY_MISMATCH", "TARGET_IS_COMPLEX"
    stage1_reason: str | None


def _compute_facade_geometry(
    poly_en: Polygon | None, street_en: np.ndarray, has_valid_street_pt: bool
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, float, float]:
    """Computes front-wall corners using the exact wall tangent `(p1 - p0) / seg_len` for `right_u`."""
    if poly_en is not None and poly_en.area >= 12.0:
        coords = list(poly_en.exterior.coords)
        c_pt = np.array([poly_en.centroid.x, poly_en.centroid.y], dtype=np.float64)
        to_street = street_en - c_pt
        to_st_norm = float(np.linalg.norm(to_street))
        curb_dir = to_street / to_st_norm if to_st_norm > 1e-3 else np.array([0.0, 1.0], dtype=np.float64)

        best_edge, best_score = None, -1e9
        st_geom = Point(float(street_en[0]), float(street_en[1]))
        for idx in range(len(coords) - 1):
            p0, p1 = np.array(coords[idx], dtype=np.float64), np.array(coords[idx + 1], dtype=np.float64)
            seg_vec = p1 - p0
            seg_len = float(np.linalg.norm(seg_vec))
            if seg_len < 2.2:
                continue
            mid = 0.5 * (p0 + p1)
            n1 = np.array([-seg_vec[1], seg_vec[0]], dtype=np.float64) / seg_len
            out_n = -n1 if poly_en.contains(Point(float(mid[0] + 0.25 * n1[0]), float(mid[1] + 0.25 * n1[1]))) else n1
            dist_st = float(LineString([p0, p1]).distance(st_geom))
            to_st_u = (street_en - mid) / max(float(np.linalg.norm(street_en - mid)), 1e-3)
            score = 14.0 * float(np.dot(out_n, to_st_u)) - 1.15 * dist_st + 0.35 * min(seg_len, 16.0)
            if score > best_score:
                best_score, best_edge = score, (p0, p1, mid, -out_n, seg_len)

        if best_edge is not None:
            p0, p1, mid, in_n, seg_len = best_edge
            wall_tan = (p1 - p0) / seg_len
            facing_right = np.array([in_n[1], -in_n[0]], dtype=np.float64)
            right_u = wall_tan if float(np.dot(wall_tan, facing_right)) >= 0.0 else -wall_tan
            f_left, f_right = (p0, p1) if float(np.dot(p0 - mid, right_u)) <= float(np.dot(p1 - mid, right_u)) else (p1, p0)

            all_pts = np.array(coords[:-1], dtype=np.float64)
            depth_along_in = (all_pts - mid) @ in_n
            front_pts = all_pts[(depth_along_in >= -3.5) & (depth_along_in <= 3.2)]
            if len(front_pts) >= 2:
                lat_proj = (front_pts - mid) @ right_u
                f_left = mid + float(lat_proj.min()) * right_u
                f_right = mid + float(lat_proj.max()) * right_u
                mid = 0.5 * (f_left + f_right)
                seg_len = float(np.linalg.norm(f_right - f_left))
            half_depth = max(2.5, min(14.0, float(np.dot(c_pt - mid, in_n))))
            return f_left, mid, f_right, in_n, half_depth, max(seg_len, 4.0)

        inward_n = -curb_dir
        front_mid = c_pt + 5.0 * curb_dir
        right_u = np.array([inward_n[1], -inward_n[0]], dtype=np.float64)
        return front_mid - 5.0 * right_u, front_mid, front_mid + 5.0 * right_u, inward_n, 5.5, 10.0

    nrm = float(np.linalg.norm(street_en))
    if has_valid_street_pt and nrm > 1.5:
        curb_dir = street_en / nrm
        front_mid = min(5.5, 0.35 * nrm) * curb_dir if nrm > 3.0 else np.zeros(2, dtype=np.float64)
    else:
        # Placeholder until Stage 2 snaps orientation to the nearest `SV_PANO` car trajectory
        curb_dir = np.array([0.0, 1.0], dtype=np.float64)
        front_mid = np.zeros(2, dtype=np.float64)
    inward_n = -curb_dir
    right_u = np.array([inward_n[1], -inward_n[0]], dtype=np.float64)
    return front_mid - 5.0 * right_u, front_mid, front_mid + 5.0 * right_u, inward_n, 5.8, 10.0


def recompute_prior_for_street_point(
    prior: ParcelPrior, street_lat: float, street_lng: float, source: str = "svi_car_trajectory"
) -> ParcelPrior:
    st_en = np.array(lla_to_en(street_lat, street_lng, prior.ref_lat, prior.ref_lng), dtype=np.float64)
    f_left, f_mid, f_right, in_n, h_depth, fac_w = _compute_facade_geometry(prior.poly_en, st_en, has_valid_street_pt=True)
    if prior.poly_en is not None:
        st_pt = Point(float(st_en[0]), float(st_en[1]))
        out_m = 0.0 if prior.poly_en.contains(st_pt) else float(prior.poly_en.distance(st_pt))
    else:
        out_m = float(np.linalg.norm(st_en))
    return dataclasses.replace(
        prior,
        nav_lat=street_lat, nav_lng=street_lng, nav_outside_poly_m=round(out_m, 2),
        needs_svi_street_ref=False, street_ref_source=source,
        front_left_en=f_left, front_mid_en=f_mid, front_right_en=f_right,
        inward_normal_en=in_n, true_half_depth_m=h_depth, facade_width_m=fac_w,
    )


def resolve_address_gmp(address: str) -> ParcelPrior:
    """Resolves parcel outline, curbside `navigationPoints`, and typed `subDestinations` via GA Geocoding API v4."""
    base_headers: dict[str, str] = {"X-Goog-User-Project": PROJECT_ID}
    if GMP_MAPS_API_KEY:
        base_headers["X-Goog-Api-Key"] = GMP_MAPS_API_KEY
    dest_headers = {**base_headers, "X-Goog-FieldMask": DESTINATIONS_FIELDMASK}

    d0: dict[str, Any] = {}
    prim: dict[str, Any] = {}
    if not _NON_PHYSICAL_ADDR_RE.match(address or ""):
        try:
            r = authed_http.post(
                GEOCODE_DESTINATIONS_URL,
                headers=dest_headers,
                json={"addressQuery": {"addressQuery": address}},
                timeout=20,
            )
            if r.status_code == 200:
                dests = r.json().get("destinations") or []
                d0 = dests[0] if dests else {}
                prim = d0.get("primary") or {}
            else:
                logger.warning("SearchDestinations HTTP %s for %r", r.status_code, address)
        except Exception as exc:
            logger.warning("SearchDestinations request failed for %r: %s", address, exc)

        if not prim:
            try:
                rf = authed_http.get(
                    f"{GEOCODE_ADDRESS_URL}/{urllib.parse.quote(address)}",
                    headers=base_headers,
                    timeout=20,
                )
                if rf.status_code == 200 and (rf.json().get("results") or []):
                    res0 = rf.json()["results"][0]
                    prim = {k: res0.get(k) for k in ("placeId", "formattedAddress", "location", "types", "addressComponents", "granularity")}
            except Exception as exc:
                logger.warning("GeocodeAddress fallback failed for %r: %s", address, exc)

    fmt_addr = str(prim.get("formattedAddress") or address)
    place_id = prim.get("placeId")
    street_num = extract_street_number(address) or extract_street_number(fmt_addr)
    route_name, route_long = None, None
    for comp in prim.get("addressComponents") or []:
        if "route" in (comp.get("types") or []):
            route_name = comp.get("shortText") or comp.get("longText")
            route_long = comp.get("longText") or route_name
        if not street_num and "street_number" in (comp.get("types") or []):
            street_num = comp.get("shortText") or comp.get("longText")

    entity_ok, entity_err = _validate_entity_match(address, prim, route_long or route_name)

    poly_geojson = prim.get("displayPolygon")
    poly_src = "primary" if poly_geojson else "none"
    cps = d0.get("containingPlaces") or []
    if not poly_geojson:
        for cp in cps:
            if cp.get("displayPolygon") and cp.get("structureType") == "BUILDING":
                poly_geojson, poly_src = cp["displayPolygon"], "containingPlaces"
                break

    dropped_lq = False
    ploc = prim.get("location") or {}
    prim_lat = float(ploc.get("latitude", 0.0)) if "latitude" in ploc else 0.0
    prim_lng = float(ploc.get("longitude", 0.0)) if "longitude" in ploc else 0.0

    if not poly_geojson and entity_ok and "latitude" in ploc and "longitude" in ploc:
        try:
            r2 = authed_http.post(
                GEOCODE_DESTINATIONS_URL,
                headers=dest_headers,
                json={"locationQuery": {"location": ploc, "placeFilter": {"structureType": "BUILDING"}}},
                timeout=20,
            )
            if r2.status_code == 200:
                for cd in r2.json().get("destinations") or []:
                    cand_gj = (cd.get("primary") or {}).get("displayPolygon")
                    if cand_gj:
                        cand_shp = shape(cand_gj)
                        if isinstance(cand_shp, MultiPolygon):
                            cand_shp = max(cand_shp.geoms, key=lambda g: g.area)
                        if isinstance(cand_shp, Polygon):
                            cand_en = Polygon([lla_to_en(y, x, prim_lat, prim_lng) for x, y in cand_shp.exterior.coords])
                            if float(cand_en.distance(Point(0.0, 0.0))) <= 2.0:
                                poly_geojson, poly_src = cand_gj, "locationQuery_BUILDING"
                            else:
                                dropped_lq = True
                        break
        except Exception as exc:
            logger.warning("locationQuery BUILDING fallback failed for %r: %s", address, exc)

    g_poly: Polygon | None = None
    if poly_geojson:
        shp = shape(poly_geojson)
        if isinstance(shp, MultiPolygon):
            shp = max(shp.geoms, key=lambda g: g.area)
        if isinstance(shp, Polygon):
            g_poly = shp

    ref_lat, ref_lng = (float(g_poly.centroid.y), float(g_poly.centroid.x)) if g_poly is not None else (prim_lat, prim_lng)
    poly_en = Polygon([lla_to_en(y, x, ref_lat, ref_lng) for x, y in g_poly.exterior.coords]) if g_poly is not None else None
    poly_area_m2 = float(poly_en.area) if poly_en is not None else 0.0

    nav_pts = d0.get("navigationPoints") or []
    has_gmp_nav = bool(nav_pts)
    if nav_pts:
        nloc = nav_pts[0].get("location") or {}
        nav_lat, nav_lng = float(nloc.get("latitude", ref_lat)), float(nloc.get("longitude", ref_lng))
        access_street = (nav_pts[0].get("displayName") or {}).get("text") or route_name
    else:
        nav_lat, nav_lng, access_street = ref_lat, ref_lng, route_name

    nav_en = np.array(lla_to_en(nav_lat, nav_lng, ref_lat, ref_lng), dtype=np.float64)
    if poly_en is not None:
        nav_geom = Point(float(nav_en[0]), float(nav_en[1]))
        nav_outside_m = 0.0 if poly_en.contains(nav_geom) else float(poly_en.distance(nav_geom))
    else:
        nav_outside_m = float(np.linalg.norm(nav_en))
    needs_svi_ref = (not has_gmp_nav) or (nav_outside_m < 2.5) or (nav_outside_m > 30.0)

    f_left, f_mid, f_right, in_n, h_depth, fac_w = _compute_facade_geometry(
        poly_en, nav_en, has_valid_street_pt=(has_gmp_nav and not needs_svi_ref)
    )

    subs = d0.get("subDestinations") or []
    unit_subs = [sd for sd in subs if bool(_UNIT_SUBDEST_TYPES & set(sd.get("types") or []))]
    has_complex_parent = any(
        cp.get("structureType") == "GROUNDS" and bool(_COMPLEX_GROUNDS_TYPES & set(cp.get("types") or []))
        for cp in cps
    )
    has_explicit_unit = bool(_EXPLICIT_UNIT_RE.search((address or "").split(",")[0]))
    is_multi_unit = (
        len(unit_subs) > 0
        or has_explicit_unit
        or has_complex_parent
        or (poly_src == "containingPlaces" and poly_area_m2 >= 650.0)
    )

    prim_struct = str(prim.get("structureType") or "")
    grounds_only_huge = False
    if poly_en is None:
        for cp in cps:
            if cp.get("structureType") == "GROUNDS" and cp.get("displayPolygon"):
                g_shp = shape(cp["displayPolygon"])
                if isinstance(g_shp, MultiPolygon):
                    g_shp = max(g_shp.geoms, key=lambda g: g.area)
                if isinstance(g_shp, Polygon):
                    g_en = Polygon([lla_to_en(y, x, ref_lat, ref_lng) for x, y in g_shp.exterior.coords])
                    if float(g_en.area) > 1500.0:
                        grounds_only_huge = True
                        break

    if not entity_ok:
        stage1_status, stage1_reason = "ENTITY_MISMATCH", entity_err
    elif has_explicit_unit and not d0:
        stage1_status, stage1_reason = "ENTITY_MISMATCH", "UNRESOLVED_UNIT_DESTINATION"
    elif prim_struct == "GROUNDS":
        stage1_status, stage1_reason = "TARGET_IS_COMPLEX", f"PRIMARY_STRUCTURE_IS_GROUNDS({poly_area_m2:.0f}m2)"
    elif poly_area_m2 > 1500.0:
        stage1_status, stage1_reason = "TARGET_IS_COMPLEX", f"BUILDING_AREA_EXCEEDS_1500M2({poly_area_m2:.0f}m2)"
    elif grounds_only_huge and has_complex_parent and not has_explicit_unit:
        stage1_status, stage1_reason = "TARGET_IS_COMPLEX", "CAMPUS_GROUNDS_WITHOUT_BUILDING"
    else:
        stage1_status, stage1_reason = "OK", None

    return ParcelPrior(
        query_address=address, formatted_address=fmt_addr, place_id=place_id,
        street_number=street_num, route_name=route_name, access_street_name=access_street,
        ref_lat=ref_lat, ref_lng=ref_lng, nav_lat=nav_lat, nav_lng=nav_lng,
        has_gmp_nav=has_gmp_nav, nav_outside_poly_m=round(nav_outside_m, 2),
        needs_svi_street_ref=needs_svi_ref,
        street_ref_source="navigationPoint" if (has_gmp_nav and not needs_svi_ref) else "geocode_fallback",
        is_multi_unit=is_multi_unit, unit_sub_destination_count=len(unit_subs),
        total_sub_destination_count=len(subs), polygon_source=poly_src,
        polygon_area_m2=round(poly_area_m2, 1), dropped_uncontained_fallback_poly=dropped_lq,
        poly_en=poly_en, front_left_en=f_left, front_mid_en=f_mid, front_right_en=f_right,
        inward_normal_en=in_n, true_half_depth_m=h_depth, facade_width_m=fac_w,
        stage1_status=stage1_status, stage1_reason=stage1_reason,
    )


prior = resolve_address_gmp(TARGET_ADDRESS)
print(f"Resolved: {prior.formatted_address} (place_id={prior.place_id}) | Stage-1: {prior.stage1_status}")
print(f"  Building outline: {prior.polygon_source} ({prior.polygon_area_m2:.0f} m^2) | width={prior.facade_width_m:.1f} m | half_depth={prior.true_half_depth_m:.1f} m")
print(f"  Street ref: {prior.street_ref_source} ({prior.access_street_name}, {prior.nav_outside_poly_m:.1f} m outside poly)")
print(f"  Multi-unit: {prior.is_multi_unit} (unit_sub_destinations={prior.unit_sub_destination_count}/{prior.total_sub_destination_count})")

## 4. Stage 2: Batched ephemeral `UNNEST(@batch_pts)` fetch & SVI-trajectory frustum selector

`pano_observations_latest` is a BigQuery view over `STRUCT` coordinates, so per-address queries would scan the entire view once per stop. Instead:
1. `fetch_panos_batch_ephemeral(priors)` passes an in-memory `ARRAY<STRUCT<idx INT64, lat FLOAT64, lng FLOAT64>>` query parameter (`UNNEST(@batch_pts)`), resolving **1 to 5,000+ stops in a single BigQuery scan** with **zero persistent GMP tables** (`ToS §3.2.3`).
2. `select_optimal_house_frames(prior, local_df)` is **pure** (`dataclasses.replace` only, returning `(selected_df, stats, active_prior)`). Whenever `navigationPoints` is missing, `< 2.5 m` outside `displayPolygon`, or `> 30 m` away, it snaps street orientation directly to the **nearest `SV_PANO` car-pass trajectory** (eliminating any North-facing default).
3. It segments captures into single car passes (`dt <= 60 s`, `|d_alt| <= 2.2 m`), prefers near-curb lenses centered near the optical axis (`cx = 1835.6`), and selects up to `MAX_PANOS = 6` viewpoints with wide angular parallax.

In [ ]:
# @title Step 4: Fisheye lens model, UNNEST(@batch_pts) fetcher & pure frustum selector
@dataclass(frozen=True)
class FisheyeLens:
    width: int = 3648
    height: int = 5472
    f_px: float = 2847.13  # approximate equidistant focal length (px/rad)
    cx: float = 1835.6
    cy: float = 2736.0
    k1: float = -0.028


LENS = FisheyeLens()


def camera_to_enu_rotation(heading_deg: float, pitch_deg: float, roll_deg: float) -> np.ndarray:
    """3x3 matrix whose columns are camera [right, down, forward] axes in local East-North-Up."""
    h, p, r = map(math.radians, (heading_deg, pitch_deg, roll_deg))
    fwd = np.array([math.sin(h) * math.cos(p), math.cos(h) * math.cos(p), math.sin(p)], dtype=np.float64)
    right0 = np.array([math.cos(h), -math.sin(h), 0.0], dtype=np.float64)
    down0 = np.cross(fwd, right0)
    right = math.cos(r) * right0 + math.sin(r) * down0
    down = -math.sin(r) * right0 + math.cos(r) * down0
    return np.stack([right, down, fwd], axis=1)


def pixel_to_camera_ray(lens: FisheyeLens, u: float, v: float) -> np.ndarray:
    mx, my = (u - lens.cx) / lens.f_px, (v - lens.cy) / lens.f_px
    theta_d = math.hypot(mx, my)
    theta = theta_d
    for _ in range(20):
        f = theta * (1.0 + lens.k1 * theta * theta) - theta_d
        theta -= f / (1.0 + 3.0 * lens.k1 * theta * theta)
    s = math.sin(theta) / theta_d if theta_d > 1e-12 else 1.0
    return np.array([mx * s, my * s, math.cos(theta)], dtype=np.float64)


def pixel_to_bearing(
    lens: FisheyeLens, heading: float, pitch: float, roll: float, u: float, v: float
) -> tuple[float, float]:
    d = camera_to_enu_rotation(heading, pitch, roll) @ pixel_to_camera_ray(lens, u, v)
    az = math.degrees(math.atan2(d[0], d[1])) % 360.0
    el = math.degrees(math.asin(max(-1.0, min(1.0, float(d[2])))))
    return az, el


def enu_offset_to_pixel(
    lens: FisheyeLens, heading: float, pitch: float, roll: float, d_east: float, d_north: float, d_up: float
) -> tuple[float, float, float] | None:
    """Exact inverse of `pixel_to_bearing`: projects camera-relative ENU `[dE, dN, dU]` to `(u_px, v_px, fwd_m)`."""
    v_cam = camera_to_enu_rotation(heading, pitch, roll).T @ np.array([d_east, d_north, d_up], dtype=np.float64)
    if v_cam[2] <= 0.25:
        return None
    norm = float(np.linalg.norm(v_cam))
    theta = math.acos(max(-1.0, min(1.0, float(v_cam[2] / norm))))
    if theta > math.radians(62.0):
        return None
    theta_d = theta * (1.0 + lens.k1 * theta * theta)
    r_xy = math.hypot(float(v_cam[0]), float(v_cam[1]))
    scale = lens.f_px * (theta_d / r_xy) if r_xy > 1e-12 else lens.f_px
    return lens.cx + float(v_cam[0]) * scale, lens.cy + float(v_cam[1]) * scale, float(v_cam[2])


def fetch_panos_batch_ephemeral(
    priors: Sequence[ParcelPrior], radius_m: float = 58.0, verbose: bool = True
) -> dict[int, pd.DataFrame]:
    """Fetches nearby `SV_PANO` frames for `N` stops in ONE BigQuery scan via in-memory `UNNEST(@batch_pts)`."""
    out_map: dict[int, pd.DataFrame] = {i: pd.DataFrame() for i in range(len(priors))}
    struct_vals: list[bigquery.StructQueryParameter] = []
    for idx, p in enumerate(priors):
        if p.stage1_status != "OK" or (p.ref_lat == 0.0 and p.ref_lng == 0.0):
            continue
        struct_vals.append(bigquery.StructQueryParameter(
            None,
            bigquery.ScalarQueryParameter("idx", "INT64", idx),
            bigquery.ScalarQueryParameter("lat", "FLOAT64", float(p.ref_lat)),
            bigquery.ScalarQueryParameter("lng", "FLOAT64", float(p.ref_lng)),
        ))
        nav_sep = distance_m(p.ref_lat, p.ref_lng, p.nav_lat, p.nav_lng)
        if p.has_gmp_nav and 2.5 <= nav_sep <= 45.0:
            struct_vals.append(bigquery.StructQueryParameter(
                None,
                bigquery.ScalarQueryParameter("idx", "INT64", idx),
                bigquery.ScalarQueryParameter("lat", "FLOAT64", float(p.nav_lat)),
                bigquery.ScalarQueryParameter("lng", "FLOAT64", float(p.nav_lng)),
            ))
    if not struct_vals:
        return out_map

    lats = [float(p.ref_lat) for p in priors if p.stage1_status == "OK" and (p.ref_lat != 0.0 or p.ref_lng != 0.0)]
    lngs = [float(p.ref_lng) for p in priors if p.stage1_status == "OK" and (p.ref_lat != 0.0 or p.ref_lng != 0.0)]
    mean_lat = sum(lats) / max(len(lats), 1)
    dlat = (radius_m + 60.0) / 111_000.0
    dlng = (radius_m + 60.0) / (111_000.0 * max(0.15, math.cos(math.radians(mean_lat))))

    sql = f"""
    WITH stops AS (
      SELECT s.idx, s.lat, s.lng, ST_GEOGPOINT(s.lng, s.lat) AS g_pt
      FROM UNNEST(@batch_pts) AS s
    ),
    prefiltered AS (
      SELECT
        o.observation_id,
        COALESCE(
          NULLIF(o.pano_id, ''),
          REGEXP_EXTRACT(o.observation_id, r'^o1:(.+)_[0-6]:5001ee$'),
          o.observation_id
        ) AS pano_id,
        CAST(REGEXP_EXTRACT(o.observation_id, r'_([0-6]):5001ee$') AS INT64) AS cam_k,
        o.capture_time,
        UNIX_MICROS(o.capture_time) AS capture_ts_us,
        o.capture_location.latitude AS capture_lat,
        o.capture_location.longitude AS capture_lng,
        o.camera_pose.latitude AS cam_lat,
        o.camera_pose.longitude AS cam_lng,
        o.camera_pose.altitude AS cam_alt,
        o.camera_pose.heading AS heading,
        o.camera_pose.pitch AS pitch,
        o.camera_pose.roll AS roll,
        o.gcs_uri
      FROM `{SVI_PROJECT_ID}.{SVI_DATASET}.pano_observations_latest` o
      WHERE o.capture_location.latitude BETWEEN @lat_min AND @lat_max
        AND o.capture_location.longitude BETWEEN @lng_min AND @lng_max
        AND REGEXP_CONTAINS(o.observation_id, r'_[0-5]:5001ee$')
        AND o.gcs_uri IS NOT NULL
    ),
    matched AS (
      SELECT DISTINCT
        s.idx AS stop_idx,
        o.*
      FROM stops s
      JOIN prefiltered o
        ON o.capture_lat BETWEEN s.lat - @stop_dlat AND s.lat + @stop_dlat
        AND o.capture_lng BETWEEN s.lng - @stop_dlng AND s.lng + @stop_dlng
        AND ST_DWITHIN(ST_GEOGPOINT(o.capture_lng, o.capture_lat), s.g_pt, @radius_m)
    )
    SELECT * FROM matched
    """
    struct_type = bigquery.StructQueryParameterType(
        bigquery.ScalarQueryParameterType("INT64", name="idx"),
        bigquery.ScalarQueryParameterType("FLOAT64", name="lat"),
        bigquery.ScalarQueryParameterType("FLOAT64", name="lng"),
    )
    params = [
        bigquery.ArrayQueryParameter("batch_pts", struct_type, struct_vals),
        bigquery.ScalarQueryParameter("radius_m", "FLOAT64", float(radius_m)),
        bigquery.ScalarQueryParameter("lat_min", "FLOAT64", min(lats) - dlat),
        bigquery.ScalarQueryParameter("lat_max", "FLOAT64", max(lats) + dlat),
        bigquery.ScalarQueryParameter("lng_min", "FLOAT64", min(lngs) - dlng),
        bigquery.ScalarQueryParameter("lng_max", "FLOAT64", max(lngs) + dlng),
        bigquery.ScalarQueryParameter("stop_dlat", "FLOAT64", dlat),
        bigquery.ScalarQueryParameter("stop_dlng", "FLOAT64", dlng),
    ]
    dry_cfg = bigquery.QueryJobConfig(query_parameters=params, dry_run=True, use_query_cache=False)
    est_gb = bq_client.query(sql, job_config=dry_cfg).total_bytes_processed / 1e9
    assert est_gb <= MAX_QUERY_GB, f"Estimated scan {est_gb:.2f} GB exceeds MAX_QUERY_GB={MAX_QUERY_GB}"
    if verbose:
        print(f"BigQuery UNNEST(@batch_pts) dry-run ({len(priors)} stop(s)): {est_gb:.2f} GB total (cap {MAX_QUERY_GB:.1f} GB)")
    run_cfg = bigquery.QueryJobConfig(query_parameters=params, maximum_bytes_billed=int(MAX_QUERY_GB * 1e9))
    df = bq_client.query(sql, job_config=run_cfg).to_dataframe()
    if not df.empty:
        df["capture_time"] = pd.to_datetime(df["capture_time"], utc=True)
        for stop_idx, grp in df.groupby("stop_idx"):
            out_map[int(stop_idx)] = grp.drop(columns=["stop_idx"]).reset_index(drop=True)
    return out_map


def compute_projected_crop_window(
    prior: ParcelPrior, cam_lat: float, cam_lng: float, heading: float, pitch: float, roll: float
) -> dict[str, Any] | None:
    cam_e, cam_n = lla_to_en(cam_lat, cam_lng, prior.ref_lat, prior.ref_lng)
    cam_pt = np.array([cam_e, cam_n], dtype=np.float64)
    nav_e, nav_n = lla_to_en(prior.nav_lat, prior.nav_lng, prior.ref_lat, prior.ref_lng)
    setback_m = float(np.linalg.norm(np.array([nav_e, nav_n]) - prior.front_mid_en))
    max_front_dist = max(48.0, min(72.0, setback_m + 24.0))

    to_cam = cam_pt - prior.front_mid_en
    dist_front = float(np.linalg.norm(to_cam))
    if not (3.2 <= dist_front <= max_front_dist):
        return None
    outward_n = -prior.inward_normal_en
    cos_inc = float(np.dot(to_cam / dist_front, outward_n))
    if cos_inc < -0.15:
        return None

    mid_proj = enu_offset_to_pixel(LENS, heading, pitch, roll, float(prior.front_mid_en[0] - cam_e), float(prior.front_mid_en[1] - cam_n), -0.4)
    if mid_proj is None or not (120.0 <= mid_proj[0] <= LENS.width - 120.0):
        return None
    u_mid, v_mid, _ = mid_proj

    pts_2d = [prior.front_left_en, prior.front_mid_en, prior.front_right_en]
    if prior.poly_en is not None:
        pts_2d.extend(np.array(c, dtype=np.float64) for c in list(prior.poly_en.exterior.coords)[:-1])
    us, vs = [], []
    for pt in pts_2d:
        de, dn = float(pt[0] - cam_e), float(pt[1] - cam_n)
        for du in (-CAM_HEIGHT_ABOVE_GROUND_M - 0.6, 0.5, 6.2):
            pr = enu_offset_to_pixel(LENS, heading, pitch, roll, de, dn, du)
            if pr is not None:
                us.append(pr[0])
                vs.append(pr[1])
    if len(us) < 2:
        return None

    raw_x0, raw_x1 = max(0.0, min(us)), min(float(LENS.width), max(us))
    raw_y0, raw_y1 = max(0.0, min(vs)), min(float(LENS.height), max(vs))
    pad_x = max(0.48 * max(raw_x1 - raw_x0, 260.0), 420.0)
    pad_y_top = max(0.40 * max(raw_y1 - raw_y0, 480.0), 360.0)
    pad_y_bot = max(0.32 * max(raw_y1 - raw_y0, 480.0), 280.0)

    cx0 = int(max(0, math.floor(min(raw_x0, u_mid - 250.0) - pad_x)))
    cx1 = int(min(LENS.width, math.ceil(max(raw_x1, u_mid + 250.0) + pad_x)))
    cy0 = int(max(0, math.floor(min(raw_y0, v_mid - 350.0) - pad_y_top)))
    cy1 = int(min(LENS.height, math.ceil(max(raw_y1, v_mid + 350.0) + pad_y_bot)))
    if cx1 - cx0 < 1280:
        h_need = (1280 - (cx1 - cx0)) // 2 + 1
        cx0, cx1 = max(0, cx0 - h_need), min(LENS.width, cx1 + h_need)
    if cy1 - cy0 < 1440:
        v_need = (1440 - (cy1 - cy0)) // 2 + 1
        cy0, cy1 = max(0, cy0 - v_need), min(LENS.height, cy1 + v_need)

    cw = max(cx1 - cx0, 1)
    az_from_house = math.degrees(math.atan2(to_cam[0], to_cam[1])) % 360.0
    out_az = math.degrees(math.atan2(outward_n[0], outward_n[1])) % 360.0
    return {
        "dist_front_m": round(dist_front, 2),
        "dist_nav_m": round(math.hypot(cam_e - nav_e, cam_n - nav_n), 2),
        "cos_incidence": round(cos_inc, 3),
        "oblique_deg": round(ang_diff_deg(az_from_house, out_az), 1),
        "u_mid_px": round(u_mid, 1),
        "v_mid_px": round(v_mid, 1),
        "crop_box": (cx0, cy0, cx1, cy1),
        "exp_x0_1000": int(np.clip(round(1000.0 * (raw_x0 - cx0) / cw), 0, 1000)),
        "exp_x1_1000": int(np.clip(round(1000.0 * (raw_x1 - cx0) / cw), 0, 1000)),
        "exp_u_mid_1000": int(np.clip(round(1000.0 * (u_mid - cx0) / cw), 0, 1000)),
    }


def _find_nearest_svi_street_point(prior: ParcelPrior, pano_meta: pd.DataFrame) -> tuple[float, float] | None:
    if pano_meta.empty:
        return None
    nav_en = np.array(lla_to_en(prior.nav_lat, prior.nav_lng, prior.ref_lat, prior.ref_lng), dtype=np.float64)
    anchor_geom = (
        Point(float(nav_en[0]), float(nav_en[1]))
        if (prior.has_gmp_nav and prior.nav_outside_poly_m <= 30.0)
        else (prior.poly_en if prior.poly_en is not None else Point(0.0, 0.0))
    )
    best_pt_en, best_dist = None, 1e9
    for _, grp in pano_meta.groupby("drive_id"):
        pts_en = [lla_to_en(float(r.cap_lat), float(r.cap_lng), prior.ref_lat, prior.ref_lng) for r in grp.itertuples()]
        valid_pts = [
            p for p in pts_en
            if prior.poly_en is None or (not prior.poly_en.contains(Point(p[0], p[1])) and prior.poly_en.distance(Point(p[0], p[1])) >= 2.5)
        ]
        if not valid_pts:
            continue
        geom = LineString(valid_pts) if len(valid_pts) >= 2 else Point(valid_pts[0])
        near_on_traj, _ = nearest_points(geom, anchor_geom)
        cand_en = np.array([near_on_traj.x, near_on_traj.y], dtype=np.float64)
        if prior.poly_en is not None and (prior.poly_en.contains(near_on_traj) or prior.poly_en.distance(near_on_traj) < 2.2):
            cand_en = min((np.array(p, dtype=np.float64) for p in valid_pts), key=lambda v: anchor_geom.distance(Point(float(v[0]), float(v[1]))))
        d_m = float(anchor_geom.distance(Point(float(cand_en[0]), float(cand_en[1]))))
        if d_m < best_dist:
            best_dist, best_pt_en = d_m, cand_en
    return en_to_lla(float(best_pt_en[0]), float(best_pt_en[1]), prior.ref_lat, prior.ref_lng) if best_pt_en is not None else None


def select_optimal_house_frames(
    prior: ParcelPrior, local_df: pd.DataFrame, max_panos: int = 6, allow_second_drive: bool = True
) -> tuple[pd.DataFrame, dict[str, Any], ParcelPrior]:
    """Pure selector returning `(selected_df, stats, active_prior)` without mutating `prior`."""
    empty_st = {"pool_frames": 0, "pool_panos": 0, "selected_panos": 0, "selected_frames": 0, "drives_used": 0, "angular_span_deg": 0.0, "frame_reduction_x": 0.0}
    if prior.stage1_status != "OK" or local_df.empty:
        return local_df.iloc[0:0].copy(), empty_st, prior

    df = local_df.copy()
    obs_stem = df["observation_id"].astype(str).str.extract(r"^o1:(.+)_[0-6]:5001ee$")[0]
    df["pano_id"] = df["pano_id"].where(df["pano_id"].notna() & (df["pano_id"].astype(str) != ""), obs_stem).fillna(df["observation_id"].astype(str))

    pano_meta = (
        df.groupby("pano_id")
        .agg(capture_time=("capture_time", "min"), cap_lat=("capture_lat", "first"), cap_lng=("capture_lng", "first"), cam_alt=("cam_alt", "median"))
        .reset_index().sort_values("capture_time").reset_index(drop=True)
    )
    drive_ids = [0]
    for idx in range(1, len(pano_meta)):
        dt = (pano_meta.loc[idx, "capture_time"] - pano_meta.loc[idx - 1, "capture_time"]).total_seconds()
        dalt = abs(float(pano_meta.loc[idx, "cam_alt"] or 0.0) - float(pano_meta.loc[idx - 1, "cam_alt"] or 0.0))
        dpos = distance_m(float(pano_meta.loc[idx, "cap_lat"]), float(pano_meta.loc[idx, "cap_lng"]), float(pano_meta.loc[idx - 1, "cap_lat"]), float(pano_meta.loc[idx - 1, "cap_lng"]))
        drive_ids.append(drive_ids[-1] + (1 if (dt > 60.0 or dalt > 2.2 or dpos > 45.0) else 0))
    pano_meta["drive_id"] = drive_ids
    drive_map = dict(zip(pano_meta["pano_id"], pano_meta["drive_id"]))

    active_prior = prior
    if active_prior.needs_svi_street_ref:
        svi_st = _find_nearest_svi_street_point(active_prior, pano_meta)
        if svi_st is not None:
            active_prior = recompute_prior_for_street_point(active_prior, svi_st[0], svi_st[1], source="svi_car_trajectory")

    drive_headings: dict[int, float | None] = {}
    for did, grp in pano_meta.groupby("drive_id"):
        if len(grp) >= 2:
            de, dn = lla_to_en(float(grp.iloc[-1]["cap_lat"]), float(grp.iloc[-1]["cap_lng"]), float(grp.iloc[0]["cap_lat"]), float(grp.iloc[0]["cap_lng"]))
            drive_headings[int(did)] = math.degrees(math.atan2(de, dn)) % 360.0 if math.hypot(de, dn) >= 3.0 else None
        else:
            drive_headings[int(did)] = None

    def _collect(cand_p: ParcelPrior) -> list[dict[str, Any]]:
        rows_out = []
        for r in df.itertuples(index=False):
            proj = compute_projected_crop_window(cand_p, float(r.cam_lat), float(r.cam_lng), float(r.heading), float(r.pitch), float(r.roll))
            if proj is None:
                continue
            did = int(drive_map[r.pano_id])
            d_hdg = drive_headings.get(did)
            same_side_bonus = 0.0
            if d_hdg is not None:
                ce, cn = lla_to_en(float(r.cam_lat), float(r.cam_lng), cand_p.ref_lat, cand_p.ref_lng)
                to_house = cand_p.front_mid_en - np.array([ce, cn])
                if 20.0 <= ang_diff_deg(math.degrees(math.atan2(to_house[0], to_house[1])) % 360.0, d_hdg) <= 160.0:
                    same_side_bonus = 2.5
            dist_f = proj["dist_front_m"]
            dist_pen = 0.0 if 6.5 <= dist_f <= 22.0 else (0.22 * (dist_f - 22.0) if dist_f > 22.0 else 0.5 * (6.5 - dist_f))
            centrality = max(0.0, 1.0 - abs(float(proj["u_mid_px"]) - LENS.cx) / (0.5 * LENS.width))
            score = 4.5 * proj["cos_incidence"] + 5.5 * centrality + same_side_bonus - dist_pen - 0.08 * max(0.0, proj["dist_nav_m"] - 18.0)
            rows_out.append({**r._asdict(), "drive_id": did, "frame_score": round(score, 3), **proj})
        return rows_out

    cand_rows = _collect(active_prior)
    by_dr: dict[int, set[str]] = {}
    for x in cand_rows:
        by_dr.setdefault(x["drive_id"], set()).add(x["pano_id"])
    best_dr_cnt = max((len(s) for s in by_dr.values()), default=0)

    if best_dr_cnt < 3 and len(pano_meta) >= 3:
        for _, d_grp in pano_meta.groupby("drive_id"):
            p_en = np.array([lla_to_en(float(r.cap_lat), float(r.cap_lng), active_prior.ref_lat, active_prior.ref_lng) for r in d_grp.itertuples()])
            ni = int(np.argmin(np.hypot(p_en[:, 0], p_en[:, 1])))
            alt_p = recompute_prior_for_street_point(active_prior, float(d_grp.iloc[ni]["cap_lat"]), float(d_grp.iloc[ni]["cap_lng"]), source="svi_car_trajectory")
            alt_rows = _collect(alt_p)
            alt_dr: dict[int, set[str]] = {}
            for x in alt_rows:
                alt_dr.setdefault(x["drive_id"], set()).add(x["pano_id"])
            cnt = max((len(s) for s in alt_dr.values()), default=0)
            if cnt > best_dr_cnt:
                best_dr_cnt, cand_rows, active_prior = cnt, alt_rows, alt_p

    if not cand_rows:
        return df.iloc[0:0].copy(), {**empty_st, "pool_frames": int(len(df)), "pool_panos": int(df["pano_id"].nunique())}, active_prior

    cdf = pd.DataFrame(cand_rows)
    best_per_pano = cdf.sort_values(["pano_id", "frame_score"], ascending=[True, False]).groupby("pano_id", as_index=False).first()
    drive_rank = (
        best_per_pano.groupby("drive_id")
        .agg(n_panos=("pano_id", "size"), n_good=("cos_incidence", lambda s: int((s >= 0.42).sum())), mean_score=("frame_score", "mean"), min_nav=("dist_nav_m", "min"), latest=("capture_time", "max"))
        .reset_index()
    )
    drive_rank["utility"] = 2.6 * np.minimum(drive_rank["n_good"], 5) + 1.2 * np.minimum(drive_rank["n_panos"], 6) + drive_rank["mean_score"] - 0.12 * np.maximum(0.0, drive_rank["min_nav"] - 14.0) + 0.35 * ((drive_rank["latest"].dt.year - 2018).clip(0, 10))
    drive_rank = drive_rank.sort_values("utility", ascending=False).reset_index(drop=True)

    def _pick_diverse(pool_df: pd.DataFrame, k_target: int, angles_so_far: list[float]) -> list[str]:
        chosen_ids, angles = [], list(angles_so_far)
        rem = pool_df.sort_values("frame_score", ascending=False).copy()
        while len(chosen_ids) < k_target and not rem.empty:
            if angles:
                ang_bonus = [min(abs(float(r.oblique_deg) - a) for a in angles) * 0.22 for r in rem.itertuples()]
                rem = rem.assign(_tot=rem["frame_score"] + ang_bonus).sort_values("_tot", ascending=False)
            top = rem.iloc[0]
            chosen_ids.append(str(top["pano_id"]))
            angles.append(float(top["oblique_deg"]))
            rem = rem[rem["pano_id"] != top["pano_id"]]
        return chosen_ids

    chosen_ids = _pick_diverse(best_per_pano[best_per_pano["drive_id"] == int(drive_rank.iloc[0]["drive_id"])], max_panos, [])
    drives_used = 1
    chosen_angles = best_per_pano.set_index("pano_id").loc[chosen_ids, "oblique_deg"].tolist()
    if allow_second_drive and len(chosen_ids) < max_panos and (len(chosen_ids) < 4 or (max(chosen_angles) - min(chosen_angles) if len(chosen_angles) >= 2 else 0.0) < 14.0) and len(drive_rank) > 1:
        for idx_d in range(1, len(drive_rank)):
            rem_slots = max_panos - len(chosen_ids)
            if rem_slots <= 0:
                break
            sec_pool = best_per_pano[(best_per_pano["drive_id"] == int(drive_rank.iloc[idx_d]["drive_id"])) & (best_per_pano["cos_incidence"] >= 0.25)]
            if sec_pool.empty:
                continue
            extra = _pick_diverse(sec_pool, min(max(5 - len(chosen_ids), 2), rem_slots), chosen_angles)
            if extra:
                chosen_ids.extend(extra)
                chosen_angles = best_per_pano.set_index("pano_id").loc[chosen_ids, "oblique_deg"].tolist()
                drives_used += 1
            if len(chosen_ids) >= 5 and (max(chosen_angles) - min(chosen_angles)) >= 16.0:
                break

    rank_map = {pid: rank_i for rank_i, pid in enumerate(chosen_ids)}
    selected = best_per_pano[best_per_pano["pano_id"].isin(chosen_ids)].copy()
    selected["selection_rank"] = selected["pano_id"].map(rank_map)
    selected = selected.sort_values(["capture_time", "cam_k"]).reset_index(drop=True)
    stats = {
        "pool_frames": int(len(df)),
        "pool_panos": int(df["pano_id"].nunique()),
        "selected_panos": int(selected["pano_id"].nunique()),
        "selected_frames": int(selected["pano_id"].size),
        "drives_used": drives_used,
        "angular_span_deg": round(float(max(chosen_angles) - min(chosen_angles)) if len(chosen_angles) >= 2 else 0.0, 1),
        "frame_reduction_x": round(float(len(df)) / max(float(len(selected)), 1.0), 1),
    }
    return selected, stats, active_prior


local_frames_df = fetch_panos_batch_ephemeral([prior])[0]
selected_frames_df, frustum_stats, prior = select_optimal_house_frames(prior, local_frames_df, max_panos=MAX_PANOS)
print(
    f"Frustum selection: {frustum_stats['pool_frames']} nearby frames ({frustum_stats['pool_panos']} rosettes) "
    f"-> {frustum_stats['selected_frames']} target-facing frames ({frustum_stats['frame_reduction_x']}x reduction, "
    f"angular span {frustum_stats['angular_span_deg']} deg across {frustum_stats['drives_used']} drive)"
)
selected_frames_df[["observation_id", "cam_k", "dist_front_m", "oblique_deg", "crop_box"]]

## 5. Stage 3: Unprimed zoom-crop detection with Gemini (Logistics & Delivery schema)

By fast-decoding (`PIL.Image.draft` 1/2-DCT) and cropping each `3648 × 5472` frame around the projected parcel window `(cx0, cy0, cx1, cy1)` (`<= 1280 px` max dimension) before calling `gemini-4-hydrogen` (`thinkingLevel="LOW"`):
* **Zero OCR Confirmation Bias & Neighbor-Plaque Guard (`ocr_mounted_on`)**: The prompt provides **only** the street name and projected horizontal pixel window `x ≈ [x0..x1]` — **never** the expected street number — and asks the model to classify whether any legible digits sit on `TARGET_BUILDING`, `TARGET_MAILBOX`, or `NEIGHBOR_HOUSE` so neighbor house plaques never falsely reject the target house.
* **Logistics-Permitted Attributes Only**: Extracts `has_porch`, `has_fence_or_gate`, `driveway_or_staging_visible`, `step_or_curb_access`, and `access_notes` (excluding residential insurance underwriting fields).
* **Lossless Sensor Mapping & Fault Isolation**: Every `0..1000` box maps losslessly back to `3648 × 5472` fisheye pixels, and per-frame exceptions are caught cleanly so transient network glitches never abort a batch.

In [ ]:
# @title Step 5: Unprimed Logistics Gemini schema, fast DCT crop helper & guarded parallel detection
class TargetHouseCropDetection(BaseModel):
    target_building_visible: bool = Field(
        description="True if the target building in the expected horizontal zone is at least partially visible (even behind trees or parked cars). False ONLY if invisible, blurred out, or non-building."
    )
    occlusion_level: str = Field(
        description="One of: CLEAR, PARTIAL_TREE_OR_VEHICLE, HEAVY_OCCLUSION, PRIVACY_BLURRED, NOT_VISIBLE"
    )
    facade_box_2d: list[int] | None = Field(
        default=None,
        description="[ymin, xmin, ymax, xmax] 0..1000 bounding the target building's street-facing front wall (never include neighbor houses or open-air gas station canopies).",
    )
    front_door_box_2d: list[int] | None = Field(
        default=None,
        description="[ymin, xmin, ymax, xmax] 0..1000 of the pedestrian front entrance door ONLY if genuinely visible (never box windows, signs, fence gates, or garage doors). Null if hidden.",
    )
    porch_or_entryway_box_2d: list[int] | None = Field(
        default=None,
        description="[ymin, xmin, ymax, xmax] 0..1000 of the front porch, recessed entry alcove, or entry steps if visible. Null if none.",
    )
    left_corner_visible: bool = Field(
        description="True if the real left vertical corner of this building's front wall is visible inside the crop."
    )
    right_corner_visible: bool = Field(
        description="True if the real right vertical corner of this building's front wall is visible inside the crop."
    )
    visible_house_number: str | None = Field(
        default=None,
        description="Exact house/building number digits visibly legible in the pixels. Report ONLY if legible; never guess. Null if unreadable.",
    )
    ocr_mounted_on: str = Field(
        default="UNKNOWN",
        description="One of: TARGET_BUILDING, TARGET_MAILBOX, NEIGHBOR_HOUSE, UNKNOWN — where visible_house_number is mounted.",
    )
    building_category: str = Field(
        description="One of: DETACHED_SINGLE_FAMILY, SEMI_DETACHED_DUPLEX, TERRACED_ROWHOUSE, MULTIFAMILY_APARTMENT, COMMERCIAL_STOREFRONT, NON_BUILDING"
    )
    has_porch: bool = Field(default=False, description="True if a covered front porch, stoop, or sheltered package drop-off area is visible.")
    has_fence_or_gate: bool = Field(default=False, description="True if a front-yard fence, gate, or access barrier crosses the pedestrian approach.")
    driveway_or_staging_visible: bool = Field(default=False, description="True if a driveway, pull-in bay, or clear curbside staging space is visible.")
    step_or_curb_access: str = Field(
        default="UNKNOWN",
        description="One of: LEVEL_OR_RAMP, STEPS_1_TO_3, STEPS_4_PLUS, OBSTRUCTED, UNKNOWN — pedestrian delivery path to entrance.",
    )
    access_notes: str | None = Field(
        default=None,
        description="Concise factual delivery-path note (<= 12 words, e.g. 'clear walkway to covered stoop'), or null. Never include personal or resident details.",
    )
    confidence: float = Field(default=0.0, description="Confidence 0.0-1.0 that facade_box_2d is the primary building in the target zone.")


def build_target_crop_prompt(prior: ParcelPrior, row: Mapping[str, Any]) -> str:
    """100% unprimed prompt — zero street-number digits appear anywhere in the prompt."""
    x0, x1, xm = int(row["exp_x0_1000"]), int(row["exp_x1_1000"]), int(row["exp_u_mid_1000"])
    street_label = prior.route_name or prior.access_street_name or "the street"
    street_label = re.sub(r"^\s*\d+(?:-\d+)?[A-Za-z]?\s+", "", str(street_label)).strip() or "the street"
    return (
        f"You are inspecting a high-resolution street-level zoom crop along '{street_label}'.\n"
        f"Camera-pose geometry projects the target parcel's street-facing building wall into horizontal region "
        f"x ≈ [{max(0, x0 - 60)}..{min(1000, x1 + 60)}] (centered near x ≈ {xm} on the 0..1000 horizontal scale).\n"
        "Instructions:\n"
        "1. Focus strictly on the primary building occupying that expected horizontal zone (do NOT box adjacent neighbor houses to the far left/right).\n"
        "2. Even if street trees, bushes, or a parked vehicle partially block the front wall, set target_building_visible=true and box the target building's street-facing wall (`facade_box_2d`).\n"
        "3. Never box an open-air gas station pump canopy or freestanding sign as the building wall; box the enclosed building structure behind it.\n"
        "4. Only populate `front_door_box_2d` when a real pedestrian entrance door is visible (never a window, garage door, fence gate, or sign). If the door is hidden behind a tree/car, leave `front_door_box_2d` null and box `porch_or_entryway_box_2d` if visible.\n"
        "5. Read `visible_house_number` ONLY if house/building number digits are genuinely legible in the pixels; otherwise return null and set `ocr_mounted_on='UNKNOWN'`. Always set `ocr_mounted_on` to `'TARGET_BUILDING'`, `'TARGET_MAILBOX'`, or `'NEIGHBOR_HOUSE'` (if the digits are visibly mounted on an adjacent neighbor house outside the target wall).\n"
        "6. If the target house in that zone is obscured by a rectangular Street View homeowner privacy blur box, set `target_building_visible=false` and `occlusion_level='PRIVACY_BLURRED'`."
    )


def crop_and_encode_jpeg(
    raw_jpeg_bytes: bytes, crop_box: tuple[int, int, int, int], max_dim: int = GEMINI_CROP_MAX_DIM
) -> bytes:
    """Fast libjpeg 1/2-DCT decode via `PIL.Image.draft` + exact scaled crop + LANCZOS downscale."""
    img = Image.open(io.BytesIO(raw_jpeg_bytes))
    orig_w, orig_h = img.size
    cx0, cy0, cx1, cy1 = crop_box
    if max(cx1 - cx0, cy1 - cy0) >= int(max_dim * 1.4):
        img.draft("RGB", (orig_w // 2, orig_h // 2))
    if img.mode != "RGB":
        img = img.convert("RGB")
    cur_w, cur_h = img.size
    sx, sy = cur_w / float(orig_w), cur_h / float(orig_h)
    sc_box = (
        max(0, int(round(cx0 * sx))),
        max(0, int(round(cy0 * sy))),
        min(cur_w, int(round(cx1 * sx))),
        min(cur_h, int(round(cy1 * sy))),
    )
    cropped = img.crop(sc_box)
    w, h = cropped.size
    if max(w, h) > max_dim:
        scale = max_dim / float(max(w, h))
        cropped = cropped.resize((int(round(w * scale)), int(round(h * scale))), Image.Resampling.LANCZOS)
    buf = io.BytesIO()
    cropped.save(buf, format="JPEG", quality=88)
    return buf.getvalue()


def map_crop_box_to_full_px(
    box_1000: Sequence[float] | None, crop_box: tuple[int, int, int, int]
) -> tuple[float, float, float, float] | None:
    if not box_1000 or len(box_1000) != 4:
        return None
    cx0, cy0, cx1, cy1 = crop_box
    cw, ch = float(cx1 - cx0), float(cy1 - cy0)
    y0, x0, y1, x1 = (float(v) for v in box_1000)
    px0, px1 = sorted((cx0 + (x0 / 1000.0) * cw, cx0 + (x1 / 1000.0) * cw))
    py0, py1 = sorted((cy0 + (y0 / 1000.0) * ch, cy0 + (y1 / 1000.0) * ch))
    px0, px1 = max(0.0, px0), min(float(LENS.width), px1)
    py0, py1 = max(0.0, py0), min(float(LENS.height), py1)
    return (px0, py0, px1, py1) if (px1 - px0) >= 8.0 and (py1 - py0) >= 8.0 else None


def detect_target_house_crops(
    prior: ParcelPrior, selected_df: pd.DataFrame, max_workers: int = 6, keep_crop_bytes: bool = True
) -> tuple[list[dict[str, Any]], dict[str, Any], dict[str, bytes]]:
    """Streams selected frames in memory, crops to the projected window, and runs Gemini in parallel."""
    empty_u = {"frames": 0, "failed_frames": 0, "input_tokens": 0, "output_tokens": 0, "thinking_tokens": 0, "est_online_cost_usd": 0.0, "est_batch_cost_usd": 0.0}
    if prior.stage1_status != "OK" or selected_df.empty:
        return [], empty_u, {}

    tls = threading.local()

    def _get_session() -> AuthorizedSession:
        sess = getattr(tls, "http", None)
        if sess is None:
            sess = AuthorizedSession(creds)
            tls.http = sess
        return sess

    def _work(row: dict[str, Any]) -> tuple[dict[str, Any], dict[str, int], bytes | None]:
        cbox = tuple(int(v) for v in row["crop_box"])
        bucket, obj = str(row["gcs_uri"]).removeprefix("gs://").split("/", 1)
        url = f"https://storage.googleapis.com/storage/v1/b/{bucket}/o/{urllib.parse.quote(obj, safe='')}?alt=media"
        resp_img = _get_session().get(url, timeout=(10, 30))
        resp_img.raise_for_status()
        crop_bytes = crop_and_encode_jpeg(resp_img.content, cbox, max_dim=GEMINI_CROP_MAX_DIM)
        prompt = build_target_crop_prompt(prior, row)
        cfg_kwargs: dict[str, Any] = {
            "temperature": 1.0,
            "max_output_tokens": 2048,
            "response_mime_type": "application/json",
            "response_schema": TargetHouseCropDetection,
            "http_options": types.HttpOptions(timeout=45_000),
        }
        if GEMINI_THINKING_LEVEL:
            cfg_kwargs["thinking_config"] = types.ThinkingConfig(thinking_level=GEMINI_THINKING_LEVEL)
        cfg = types.GenerateContentConfig(**cfg_kwargs)
        for attempt in range(5):
            try:
                resp = genai_client.models.generate_content(
                    model=GEMINI_MODEL,
                    contents=[types.Part.from_bytes(data=crop_bytes, mime_type="image/jpeg"), prompt],
                    config=cfg,
                )
                parsed = (
                    resp.parsed
                    if isinstance(resp.parsed, TargetHouseCropDetection)
                    else TargetHouseCropDetection.model_validate_json(resp.text or "{}")
                )
                um = resp.usage_metadata
                usage = {
                    "in": int(getattr(um, "prompt_token_count", 0) or 0),
                    "out": int(getattr(um, "candidates_token_count", 0) or 0),
                    "thinking": int(getattr(um, "thoughts_token_count", 0) or 0),
                }
                det = {"observation_id": row["observation_id"], "pano_id": row["pano_id"], "cam_k": int(row["cam_k"]), "crop_box": list(cbox), **parsed.model_dump()}
                return det, usage, (crop_bytes if keep_crop_bytes else None)
            except ValidationError:
                raise
            except Exception as exc:
                code_ = getattr(exc, "code", None)
                retryable = code_ in (408, 429, 500, 502, 503, 504) or isinstance(exc, (TimeoutError, ConnectionError))
                if attempt == 4 or not retryable:
                    raise
                time.sleep(min(40.0, (2.0 ** attempt) + random.random()))
        raise RuntimeError("Unreachable")

    records = selected_df.to_dict("records")
    dets: list[dict[str, Any]] = []
    crops_mem: dict[str, bytes] = {}
    tot = {"in": 0, "out": 0, "thinking": 0}
    failed = 0
    with cf.ThreadPoolExecutor(max_workers=max_workers) as pool:
        futs = [pool.submit(_work, r) for r in records]
        for f in cf.as_completed(futs):
            try:
                d, u, cb = f.result()
                dets.append(d)
                if cb is not None:
                    crops_mem[d["observation_id"]] = cb
                for k in tot:
                    tot[k] += u[k]
            except Exception as exc:
                failed += 1
                logger.warning("Crop detection failed on frame: %s: %s", type(exc).__name__, str(exc)[:140])

    order_map = {r["observation_id"]: i for i, r in enumerate(records)}
    dets.sort(key=lambda x: order_map.get(x["observation_id"], 999))
    online_usd = (tot["in"] * GEMINI_ONLINE_INPUT_USD_PER_M + (tot["out"] + tot["thinking"]) * GEMINI_ONLINE_OUTPUT_USD_PER_M) / 1e6
    batch_usd = (tot["in"] * GEMINI_BATCH_INPUT_USD_PER_M + (tot["out"] + tot["thinking"]) * GEMINI_BATCH_OUTPUT_USD_PER_M) / 1e6
    summary = {
        "frames": len(records), "failed_frames": failed,
        "input_tokens": tot["in"], "output_tokens": tot["out"], "thinking_tokens": tot["thinking"],
        "est_online_cost_usd": round(online_usd, 4), "est_batch_cost_usd": round(batch_usd, 4),
    }
    return dets, summary, crops_mem


t0 = time.time()
crop_detections, gemini_usage, crop_images_mem = detect_target_house_crops(prior, selected_frames_df)
print(f"Gemini ({GEMINI_MODEL}, thinking={GEMINI_THINKING_LEVEL}) finished in {time.time() - t0:.1f}s: {gemini_usage}")
if crop_detections:
    display_cols = ["observation_id", "target_building_visible", "occlusion_level", "visible_house_number", "ocr_mounted_on", "step_or_curb_access", "driveway_or_staging_visible", "access_notes"]
    display(pd.DataFrame(crop_detections)[display_cols])

## 6. Stage 4, Stage 5 & Decoupled House vs. Conditional Front-Door Calibration

* **Stage 4A — Use Case 1: Address $\rightarrow$ House Detection (`persistable_house=True`)**:
  Solves **all** building rays (`door`, `porch`, `facade_center`) via **Cauchy RANSAC (`0.85 m` kernel)** + post-IRLS outlier trimming (`<= 1.60 m`) + Huber-weighted RMS, applies loose 3D `z_agl` elevation sanity (demoting 2nd-story windows), ignores neighbor-mounted OCR plaques (`ocr_mounted_on == 'NEIGHBOR_HOUSE'`), enforces `T4-safe` homeowner privacy-blur gating, and scores `compute_house_confidence()` with **zero door-visibility penalty** (a clearly detected house whose door is behind a tree stays `GREEN`).
* **Stage 4B — Use Case 2: Conditional Front-Door Localization (`persistable_door=True`)**:
  Runs **only** when Stage 4A confirms a house (`house_zone != RED`), solving **strictly pure `door` rays** (`DOOR_LOCKED_3PLUS`, `DOOR_LOCKED_2RAY`, or explicit `PORCH_FALLBACK`) and reporting `DOOR_SINGLE_VIEW` / `DOOR_OCCLUDED` (`door_lat=None`, zero false door pin) when foliage or parked vehicles hide the door. An optional ephemeral wall-clamped display point (`door_display_ephemeral_lat/lng`, `persistable=False`, 0-day RAM only) is computed separately for interactive UI rendering.
* **Stage 5 — Surgical `pano_lidar_metadata_latest` Wall Verification**:
  Joins `pano_observations_latest` + `pano_lidar_metadata_latest` (`SV_PANO` + 3D LiDAR SKU) with a dry-run byte check and promotes a `TIER_B` house **only** when `>= 2` RGB rays (`>= 12°` parallax) agree within `<= 1.0 m` on the **farthest** qualifying vertical planar wall slab (`<= 25°` normal alignment), reporting `lidar_plane_rms_m` separately from `ray_rms_m`.

In [ ]:
# @title Step 6: Decoupled Cauchy RANSAC House & Door solvers, surgical 3D LiDAR gate & dual confidence model
FEATURE_KEYS = [
    "has_porch",
    "has_fence_or_gate",
    "driveway_or_staging_visible",
    "step_or_curb_access",
    "access_notes",
]


@dataclass
class TargetRayView:
    observation_id: str
    pano_id: str
    cam_k: int
    gcs_uri: str
    cam_lat: float
    cam_lng: float
    cam_alt: float
    cam_en: np.ndarray
    az_key_deg: float
    el_key_deg: float
    keypoint_type: str
    facade_px: tuple[float, float, float, float]
    door_px: tuple[float, float, float, float] | None
    porch_px: tuple[float, float, float, float] | None
    z_agl_m: float
    weight: float
    ocr_number: str | None
    ocr_match_type: str
    ocr_mounted_on: str
    occlusion_level: str
    raw_det: dict[str, Any]


def classify_house_number_ocr(
    ocr_raw: str | None,
    target_raw: str | None,
    is_multi_unit: bool,
    is_shared_duplex_view: bool = False,
    ocr_mounted_on: str | None = None,
) -> tuple[str, str | None]:
    """Exact token/range house-number matcher (`EXACT_MATCH`, `SIBLING_UNIT`, `CONFLICT`, `UNREAD`)."""
    if str(ocr_mounted_on or "").upper() == "NEIGHBOR_HOUSE":
        return "UNREAD", None
    if not ocr_raw or not str(ocr_raw).strip():
        return "UNREAD", None
    ocr_tokens = [t.upper() for t in re.findall(r"[NSEW]?\d+[A-Za-z]?", str(ocr_raw).strip())]
    if not ocr_tokens:
        return "UNREAD", None
    ocr_primary = ocr_tokens[0]
    if not target_raw or not str(target_raw).strip():
        return "UNREAD", ocr_primary

    t_str = str(target_raw).strip().upper()
    valid_exact: set[str] = set()
    m_range = re.match(r"^(\d+)-(\d+)$", t_str)
    if m_range:
        a, b = int(m_range.group(1)), int(m_range.group(2))
        valid_exact.update({str(a), str(b)})
        if a < b and (b - a) <= 20:
            for k in range(a, b + 1, 2 if (a % 2 == b % 2) else 1):
                valid_exact.add(str(k))
    else:
        for tok in re.findall(r"[NSEW]?\d+[A-Za-z]?", t_str):
            valid_exact.add(tok.upper())
            m_base = re.match(r"^([NSEW]?\d+)[A-Z]$", tok.upper())
            if m_base and is_multi_unit:
                valid_exact.add(m_base.group(1))

    for tok in ocr_tokens:
        if tok in valid_exact:
            return "EXACT_MATCH", tok
        m_obase = re.match(r"^([NSEW]?\d+)[A-Z]$", tok)
        if m_obase and is_multi_unit and m_obase.group(1) in valid_exact:
            return "SIBLING_UNIT", tok

    if is_multi_unit or is_shared_duplex_view:
        t_dig, o_dig = re.match(r"^(\d+)", t_str), re.match(r"^(\d+)", ocr_primary)
        max_step = 6 if is_multi_unit else 2
        if t_dig and o_dig and 0 < abs(int(t_dig.group(1)) - int(o_dig.group(1))) <= max_step:
            return "SIBLING_UNIT", ocr_primary
    return "CONFLICT", ocr_primary


def build_target_ray_views(
    prior: ParcelPrior, selected_df: pd.DataFrame, detections: Sequence[Mapping[str, Any]]
) -> tuple[list[TargetRayView], dict[str, Any]]:
    records = selected_df.to_dict("records")
    pose_by_obs = {r["observation_id"]: r for r in records}
    alts = sorted(float(r.get("cam_alt") or 0.0) for r in records)
    med_alt = alts[len(alts) // 2] if alts else 0.0

    views: list[TargetRayView] = []
    conflict_numbers: list[str] = []

    for d in detections:
        occ = str(d.get("occlusion_level") or "CLEAR")
        if occ in ("NOT_VISIBLE", "PRIVACY_BLURRED"):
            continue
        if not d.get("target_building_visible") or d.get("building_category") == "NON_BUILDING":
            continue
        p = pose_by_obs.get(d["observation_id"])
        if not p:
            continue
        cbox = tuple(int(v) for v in d["crop_box"])
        facade_px = map_crop_box_to_full_px(d.get("facade_box_2d"), cbox)
        if facade_px is None:
            continue
        door_px = map_crop_box_to_full_px(d.get("front_door_box_2d"), cbox)
        porch_px = map_crop_box_to_full_px(d.get("porch_or_entryway_box_2d"), cbox)

        shared_dup_vw = (
            prior.polygon_source == "containingPlaces"
            and d.get("building_category") in ("SEMI_DETACHED_DUPLEX", "MULTIFAMILY_APARTMENT", "TERRACED_ROWHOUSE")
        )
        mounted_on = str(d.get("ocr_mounted_on") or "UNKNOWN").upper()
        ocr_status, ocr_norm = classify_house_number_ocr(
            d.get("visible_house_number"),
            prior.street_number,
            prior.is_multi_unit,
            is_shared_duplex_view=shared_dup_vw,
            ocr_mounted_on=mounted_on,
        )
        if ocr_status == "CONFLICT":
            if ocr_norm:
                conflict_numbers.append(ocr_norm)
            continue
        if ocr_status == "SIBLING_UNIT":
            door_px = None

        fx0, fy0, fx1, fy1 = facade_px
        fh = max(fy1 - fy0, 10.0)
        l_vis = bool(d.get("left_corner_visible", True)) and (fx0 - cbox[0]) > 22.0
        r_vis = bool(d.get("right_corner_visible", True)) and (cbox[2] - fx1) > 22.0

        if door_px is not None:
            u_key, v_key, k_type, base_w = 0.5 * (door_px[0] + door_px[2]), door_px[1] + 0.82 * (door_px[3] - door_px[1]), "door", 1.45
        elif porch_px is not None:
            u_key, v_key, k_type, base_w = 0.5 * (porch_px[0] + porch_px[2]), porch_px[1] + 0.80 * (porch_px[3] - porch_px[1]), "porch", 1.15
        else:
            if l_vis and r_vis:
                u_key, base_w = 0.5 * (fx0 + fx1), 0.90
            else:
                u_key = float(np.clip(0.65 * float(p["u_mid_px"]) + 0.35 * (0.5 * (fx0 + fx1)), fx0 + 0.15 * (fx1 - fx0), fx1 - 0.15 * (fx1 - fx0)))
                base_w = 0.65
            v_key, k_type = fy0 + 0.78 * fh, "facade_center"

        hdg, pitch, roll = float(p["heading"]), float(p["pitch"]), float(p["roll"])
        az_k, el_k = pixel_to_bearing(LENS, hdg, pitch, roll, u_key, v_key)
        o_en = np.array(lla_to_en(float(p["cam_lat"]), float(p["cam_lng"]), prior.ref_lat, prior.ref_lng), dtype=np.float64)
        cam_alt = float(p.get("cam_alt") or med_alt)
        d_vec = np.array([math.sin(math.radians(az_k)), math.cos(math.radians(az_k))], dtype=np.float64)
        n_vec = np.array([math.cos(math.radians(az_k)), -math.sin(math.radians(az_k))], dtype=np.float64)

        along_mid = float((prior.front_mid_en - o_en) @ d_vec)
        if not (MIN_RAY_RANGE_M <= along_mid <= MAX_RAY_RANGE_M):
            continue
        dist_poly = (
            float(prior.poly_en.distance(LineString([o_en, o_en + (MAX_RAY_RANGE_M + 3.0) * d_vec])))
            if prior.poly_en is not None
            else abs(float((prior.front_mid_en - o_en) @ n_vec))
        )
        ocr_exact = ocr_status == "EXACT_MATCH"
        if dist_poly > (8.5 if ocr_exact else 5.5):
            continue

        # Loose 3D elevation sanity (demotes obvious 2nd-story windows back to facade_center)
        z_agl = CAM_HEIGHT_ABOVE_GROUND_M + (cam_alt - med_alt) + along_mid * math.tan(math.radians(el_k))
        z_max = 6.5 if k_type in ("door", "porch") else 8.5
        if z_agl < -3.5 or z_agl > z_max:
            if k_type == "door":
                k_type, base_w, door_px = "facade_center", 0.70, None
                u_key, v_key = 0.5 * (fx0 + fx1), fy0 + 0.78 * fh
                az_k, el_k = pixel_to_bearing(LENS, hdg, pitch, roll, u_key, v_key)
                z_agl = CAM_HEIGHT_ABOVE_GROUND_M + (cam_alt - med_alt) + along_mid * math.tan(math.radians(el_k))

        occ_mult = 1.0 if occ == "CLEAR" else (0.82 if occ == "PARTIAL_TREE_OR_VEHICLE" else 0.55)
        raw_conf = d.get("confidence")
        conf = max(0.35, min(1.0, float(raw_conf) if raw_conf is not None else 0.75))
        w = base_w * occ_mult * conf * (1.90 if ocr_exact else (0.78 if ocr_status == "SIBLING_UNIT" else 1.0))

        views.append(TargetRayView(
            observation_id=str(d["observation_id"]), pano_id=str(d["pano_id"]), cam_k=int(d["cam_k"]),
            gcs_uri=str(p["gcs_uri"]), cam_lat=float(p["cam_lat"]), cam_lng=float(p["cam_lng"]),
            cam_alt=cam_alt, cam_en=o_en, az_key_deg=az_k, el_key_deg=el_k, keypoint_type=k_type,
            facade_px=facade_px, door_px=door_px, porch_px=porch_px, z_agl_m=round(z_agl, 2), weight=w,
            ocr_number=ocr_norm, ocr_match_type=ocr_status, ocr_mounted_on=mounted_on,
            occlusion_level=occ, raw_det=dict(d),
        ))
    return views, {"conflict_count": len(conflict_numbers), "conflict_numbers": conflict_numbers}


def _aggregate_logistics_features(views: Sequence[TargetRayView]) -> tuple[dict[str, Any], dict[str, str]]:
    feats, agree = {}, {}
    n = max(len(views), 1)
    ordered = sorted(views, key=lambda v: -v.weight)
    for k in FEATURE_KEYS:
        vals = [v.raw_det.get(k) for v in ordered if v.raw_det.get(k) not in (None, "", "UNKNOWN", "NOT_VISIBLE")]
        if not vals:
            feats[k], agree[k] = None, f"0/{n}"
        elif k == "access_notes":
            feats[k], agree[k] = str(vals[0]).strip(), f"{len(vals)}/{n}"
        elif isinstance(vals[0], bool):
            t_cnt = sum(bool(x) for x in vals)
            chosen_b = t_cnt >= (len(vals) - t_cnt)
            feats[k], agree[k] = chosen_b, f"{t_cnt if chosen_b else (len(vals) - t_cnt)}/{n}"
        else:
            s_vals = [str(x) for x in vals]
            top = max(set(s_vals), key=s_vals.count)
            feats[k], agree[k] = top, f"{s_vals.count(top)}/{n}"
    return feats, agree


def _ephemeral_building_centroid(prior: ParcelPrior, wall_en: np.ndarray, wall_normal_en: np.ndarray) -> tuple[float, float]:
    cent_en = wall_en + prior.true_half_depth_m * wall_normal_en
    if prior.poly_en is not None and not prior.poly_en.contains(Point(float(cent_en[0]), float(cent_en[1]))):
        inter_p = LineString([wall_en - 2.0 * wall_normal_en, wall_en + 18.0 * wall_normal_en]).intersection(prior.poly_en)
        if not inter_p.is_empty:
            cent_en = np.array([inter_p.centroid.x, inter_p.centroid.y], dtype=np.float64)
    c_lat, c_lng = en_to_lla(float(cent_en[0]), float(cent_en[1]), prior.ref_lat, prior.ref_lng)
    return round(c_lat, 7), round(c_lng, 7)


def _clamp_point_to_polygon_ephemeral(
    prior: ParcelPrior, pt_en: np.ndarray, max_shift_m: float = 3.5
) -> tuple[float, float] | None:
    """Ephemeral 0-day UI wall clamp (`persistable=False`; never stored to BigQuery or disk)."""
    if prior.poly_en is None:
        return None
    pt_geom = Point(float(pt_en[0]), float(pt_en[1]))
    near_on_wall, _ = nearest_points(prior.poly_en.exterior, pt_geom)
    if float(near_on_wall.distance(pt_geom)) > max_shift_m:
        return None
    lat, lng = en_to_lla(float(near_on_wall.x), float(near_on_wall.y), prior.ref_lat, prior.ref_lng)
    return round(lat, 7), round(lng, 7)


def _intersect_ray_with_prior_ephemeral(prior: ParcelPrior, o_en: np.ndarray, d_vec: np.ndarray) -> np.ndarray | None:
    ray_line = LineString([o_en + 2.0 * d_vec, o_en + MAX_RAY_RANGE_M * d_vec])
    if prior.poly_en is not None:
        inter = ray_line.intersection(prior.poly_en.exterior)
        pts = [np.array([inter.x, inter.y])] if isinstance(inter, Point) else [np.array([g.x, g.y]) for g in getattr(inter, "geoms", []) if isinstance(g, Point)]
        if pts:
            return min(pts, key=lambda p: float(np.linalg.norm(p - o_en)))
        np_poly, np_ray = nearest_points(prior.poly_en.exterior, ray_line)
        if np_poly.distance(np_ray) <= 4.5:
            return np.array([np_poly.x, np_poly.y], dtype=np.float64)
    denom = float(np.dot(d_vec, prior.inward_normal_en))
    if abs(denom) > 0.12:
        t = float(np.dot(prior.front_mid_en - o_en, prior.inward_normal_en)) / denom
        if MIN_RAY_RANGE_M <= t <= MAX_RAY_RANGE_M:
            return o_en + t * d_vec
    return None


def _pure_huber_ray_solve(
    Oi: np.ndarray, Di: np.ndarray, Ni: np.ndarray, Wi: np.ndarray
) -> tuple[np.ndarray | None, float, bool]:
    """Pure camera-ray Huber-IRLS intersection (`0` GMP polygon coordinates in `A x = b`, Huber-weighted RMS)."""
    try:
        pt_est = np.linalg.solve((Ni.T * Wi) @ Ni, (Ni.T * Wi) @ np.sum(Ni * Oi, axis=1))
    except np.linalg.LinAlgError:
        return None, 999.0, False
    w_eff = Wi.copy()
    for _ in range(8):
        ranges = np.maximum(4.0, np.linalg.norm(pt_est - Oi, axis=1))
        inv_r2 = 1.0 / ((0.18 + 0.025 * ranges) ** 2)
        res = np.abs(np.sum((pt_est - Oi) * Ni, axis=1))
        w_eff = Wi * inv_r2 * np.where(res <= 1.1, 1.0, 1.1 / np.maximum(res, 1e-6))
        try:
            pt_est = np.linalg.solve((Ni.T * w_eff) @ Ni, (Ni.T * w_eff) @ np.sum(Ni * Oi, axis=1))
        except np.linalg.LinAlgError:
            return None, 999.0, False
    along_chk = np.sum((pt_est - Oi) * Di, axis=1)
    perp_sq = np.sum((pt_est - Oi) * Ni, axis=1) ** 2
    rms_m = float(np.sqrt(np.average(perp_sq, weights=np.maximum(w_eff, 1e-9))))
    return pt_est, rms_m, bool(np.all((along_chk >= MIN_RAY_RANGE_M) & (along_chk <= MAX_RAY_RANGE_M)))


def _cauchy_ransac_and_solve(
    prior: ParcelPrior,
    uviews: Sequence[TargetRayView],
    weights: Sequence[float],
    gate_m: float,
) -> tuple[list[TargetRayView], np.ndarray | None, float, float, bool]:
    """Cauchy quality RANSAC (`0.85 m` kernel) + post-IRLS outlier trimming (`<= 1.60 m` when >=3 rays remain)."""
    m = len(uviews)
    if m == 0:
        return [], None, 999.0, 0.0, False
    if m == 1:
        return [uviews[0]], None, 999.0, 0.0, False

    O = np.array([v.cam_en for v in uviews], dtype=np.float64)
    AZ = np.array([v.az_key_deg for v in uviews], dtype=np.float64)
    D = np.column_stack([np.sin(np.radians(AZ)), np.cos(np.radians(AZ))])
    N = np.column_stack([np.cos(np.radians(AZ)), -np.sin(np.radians(AZ))])
    W = np.array(weights, dtype=np.float64)

    seeds: list[np.ndarray] = []
    for i in range(m):
        for j in range(i + 1, m):
            if angle_between_rays_deg(float(AZ[i]), float(AZ[j])) < 4.5:
                continue
            try:
                pt = np.linalg.solve(np.vstack([N[i], N[j]]), np.array([N[i] @ O[i], N[j] @ O[j]]))
            except np.linalg.LinAlgError:
                continue
            if MIN_RAY_RANGE_M <= float((pt - O[i]) @ D[i]) <= MAX_RAY_RANGE_M and MIN_RAY_RANGE_M <= float((pt - O[j]) @ D[j]) <= MAX_RAY_RANGE_M:
                if prior.poly_en is None or prior.poly_en.distance(Point(float(pt[0]), float(pt[1]))) <= 7.0:
                    seeds.append(pt)
    if not seeds:
        for i in range(m):
            hi = _intersect_ray_with_prior_ephemeral(prior, O[i], D[i])
            if hi is not None:
                seeds.append(hi)

    best_inl, best_sc = list(range(min(m, 2))), (-1, -1e9)
    for s_pt in seeds:
        rel = s_pt - O
        along, perp = np.sum(rel * D, axis=1), np.abs(np.sum(rel * N, axis=1))
        inl = [k for k in range(m) if MIN_RAY_RANGE_M <= along[k] <= MAX_RAY_RANGE_M and perp[k] <= gate_m]
        if not inl:
            continue
        cauchy = 1.0 / (1.0 + (perp[inl] / 0.85) ** 2)
        qual = float(np.sum(W[inl] * cauchy))
        pen = 0.20 * float(prior.poly_en.distance(Point(float(s_pt[0]), float(s_pt[1])))) if prior.poly_en is not None else 0.06 * float(np.linalg.norm(s_pt - prior.front_mid_en))
        sc = (len(inl) >= 3, qual - pen)
        if sc > best_sc:
            best_sc, best_inl = sc, inl

    Oi, Di, Ni, Wi = O[best_inl], D[best_inl], N[best_inl], W[best_inl]
    pt_est, rms_m, valid_front = _pure_huber_ray_solve(Oi, Di, Ni, Wi)
    if pt_est is not None and len(best_inl) >= 4:
        perp_i = np.abs(np.sum((pt_est - Oi) * Ni, axis=1))
        trimmed = [best_inl[idx_k] for idx_k in range(len(best_inl)) if perp_i[idx_k] <= 1.60]
        if 3 <= len(trimmed) < len(best_inl):
            best_inl = trimmed
            Oi, Di, Ni, Wi = O[best_inl], D[best_inl], N[best_inl], W[best_inl]
            pt_est, rms_m, valid_front = _pure_huber_ray_solve(Oi, Di, Ni, Wi)

    chosen = [uviews[k] for k in best_inl]
    Az_i = AZ[best_inl]
    max_par = max((angle_between_rays_deg(float(Az_i[a]), float(Az_i[b])) for a in range(len(Az_i)) for b in range(a + 1, len(Az_i))), default=0.0)
    return chosen, pt_est, rms_m, max_par, valid_front


def _is_privacy_blurred_t4_safe(
    prior: ParcelPrior,
    raw_detections: Sequence[Mapping[str, Any]],
    views: Sequence[TargetRayView],
) -> tuple[bool, int]:
    """`T4-safe`: preserves 100% real homeowner blurs while avoiding neighbor-blur false REDs."""
    n_total = len(raw_detections)
    if n_total == 0:
        return False, 0

    target_blur_cnt = 0
    for d in raw_detections:
        if d.get("occlusion_level") != "PRIVACY_BLURRED":
            continue
        if str(d.get("ocr_mounted_on") or "UNKNOWN").upper() == "NEIGHBOR_HOUSE":
            continue
        ocr_st, _ = classify_house_number_ocr(d.get("visible_house_number"), prior.street_number, prior.is_multi_unit)
        if ocr_st == "CONFLICT":
            continue
        target_blur_cnt += 1

    if target_blur_cnt == 0:
        return False, 0

    exact_views = sum(1 for v in views if v.ocr_match_type == "EXACT_MATCH")
    if n_total >= 4 and (target_blur_cnt / float(n_total)) < 0.50 and len({v.pano_id for v in views}) >= 3 and exact_views >= 2:
        return False, target_blur_cnt

    is_blurred = (
        target_blur_cnt >= 2
        or (n_total >= 2 and (target_blur_cnt / float(n_total)) >= 0.34)
        or (len(views) == 0 and target_blur_cnt >= 1)
    )
    return is_blurred, target_blur_cnt


def compute_house_confidence(
    house_res: Mapping[str, Any], prior: ParcelPrior, operating_point: str = CONFIDENCE_OPERATING_POINT
) -> dict[str, Any]:
    """Use Case 1 (Address -> House Detection) score (0-100) and zone (`GREEN`/`YELLOW`/`RED`) — ZERO door penalty."""
    reasons: list[str] = []
    if prior.stage1_status == "ENTITY_MISMATCH":
        return {"confidence_score": 0, "confidence_zone": "RED", "confidence_reasons": [f"STAGE1_ENTITY_MISMATCH:{prior.stage1_reason}"]}
    if prior.stage1_status == "TARGET_IS_COMPLEX":
        return {"confidence_score": 10, "confidence_zone": "RED", "confidence_reasons": [f"STAGE1_TARGET_IS_COMPLEX:{prior.stage1_reason}"]}
    if house_res.get("privacy_blurred") or house_res.get("status") == "PRIVACY_BLURRED_IN_SVI":
        return {"confidence_score": 0, "confidence_zone": "RED", "confidence_reasons": ["PRIVACY_BLURRED_IN_SVI"]}
    if house_res.get("tier") in ("MISSED", "REJECTED", None) or house_res.get("house_lat") is None:
        tag = "OCR_CONFLICT_NO_VALID_VIEWS" if house_res.get("ocr_verification_status") == "CONFLICT" else "NO_USABLE_SVI_VIEWS"
        return {"confidence_score": 5, "confidence_zone": "RED", "confidence_reasons": [tag]}

    ocr_stat = str(house_res.get("ocr_verification_status") or "UNREAD")
    n_pos = int(house_res.get("n_positions") or 0)
    rms = house_res.get("ray_rms_m")
    par = float(house_res.get("max_parallax_deg") or 0.0)
    views: list[TargetRayView] = list(house_res.get("views_used") or [])

    # `T1-TierA`: Converged TIER_A (`>=3` pos, `rms <= 1.20 m`) OCR conflict -> YELLOW; unconverged -> RED
    tier_a_conflict_yellow = False
    if ocr_stat == "CONFLICT":
        if house_res.get("tier") == "TIER_A" and n_pos >= 3 and rms is not None and float(rms) <= 1.20:
            tier_a_conflict_yellow = True
            reasons.append("OCR_CONFLICT_ON_CONVERGED_HOUSE_YELLOW_CAP")
        else:
            return {"confidence_score": 18, "confidence_zone": "RED", "confidence_reasons": ["UNPRIMED_OCR_HOUSE_NUMBER_CONFLICT"]}

    score = 0.0
    score += 30.0 if n_pos >= 4 else (24.0 if n_pos == 3 else (12.0 if n_pos == 2 else 4.0))
    reasons.append(f"CONVERGED_{n_pos}_CAR_POSITIONS" if n_pos >= 3 else f"LIMITED_{n_pos}_CAR_POSITIONS")
    if rms is not None:
        rf = float(rms)
        score += 18.0 if rf <= 0.45 else (13.0 if rf <= 0.95 else (8.0 if rf <= 1.35 else (4.0 if rf <= MAX_TIER_A_RMS_M else 0.0)))
    score += 10.0 if par >= 30.0 else (7.0 if par >= 18.0 else (4.0 if par >= 12.0 else 0.0))

    exact_cnt = sum(1 for v in views if v.ocr_match_type == "EXACT_MATCH")
    if ocr_stat == "EXACT_MATCH":
        score += 20.0 if exact_cnt >= 2 else 16.0
        reasons.append(f"UNPRIMED_OCR_EXACT_MATCH({exact_cnt}_VIEWS)")
    elif ocr_stat == "SIBLING_UNIT":
        score += 2.0
        reasons.append("SIBLING_UNIT_OCR_OBSERVED")
    elif ocr_stat == "CONFLICT":
        score -= 18.0
    else:
        score += 10.0

    if int(house_res.get("ocr_conflict_count") or 0) > 0 and ocr_stat != "EXACT_MATCH":
        score -= 8.0
        reasons.append("REJECTED_NEIGHBOR_OCR_FRAME_IN_POOL")

    score += {"primary": 12.0, "containingPlaces": 9.0, "locationQuery_BUILDING": 8.0}.get(prior.polygon_source, 0.0)
    score += 5.0 if prior.street_ref_source == "navigationPoint" else (4.0 if prior.street_ref_source == "svi_car_trajectory" else 0.0)
    clear_cnt = sum(1 for v in views if v.occlusion_level == "CLEAR")
    score += 5.0 if clear_cnt >= 2 else (3.0 if views and any(v.occlusion_level != "HEAVY_OCCLUSION" for v in views) else 0.0)

    hard_yellow_cap = tier_a_conflict_yellow
    if house_res.get("tier") != "TIER_A" or not house_res.get("persistable_house"):
        hard_yellow_cap = True
        reasons.append("TIER_B_FEWER_THAN_3_RAYS_CAP")
    if prior.polygon_source == "none":
        hard_yellow_cap = True
        reasons.append("MISSING_DISPLAY_POLYGON_CAP")
    if prior.is_multi_unit and ocr_stat != "EXACT_MATCH" and operating_point != "high_recall":
        hard_yellow_cap = True
        reasons.append("MULTI_UNIT_SHARED_STRUCTURE_CAP")
    if prior.polygon_source == "containingPlaces" and prior.polygon_area_m2 >= 260.0 and ocr_stat != "EXACT_MATCH" and operating_point != "high_recall":
        hard_yellow_cap = True
        reasons.append("SHARED_CONTAINING_BUILDING_WITHOUT_OCR_CAP")
    if par < 14.0 and not house_res.get("lidar_used") and operating_point != "high_recall":
        hard_yellow_cap = True
        reasons.append("LOW_PARALLAX_DEPTH_UNCERTAINTY_CAP")

    if operating_point == "strict":
        green_thresh = 80
        if not (ocr_stat == "EXACT_MATCH" or (n_pos >= 4 and rms is not None and float(rms) <= 0.65 and not prior.is_multi_unit)):
            hard_yellow_cap = True
            reasons.append("STRICT_MODE_REQUIRES_OCR_OR_4_TIGHT_HOUSE_RAYS")
    elif operating_point == "high_recall":
        green_thresh = 60
    else:
        green_thresh = 70

    final_score = int(round(max(12.0, min(100.0, score))))
    if hard_yellow_cap and final_score >= green_thresh:
        final_score = green_thresh - 2
    zone = "GREEN" if (final_score >= green_thresh and not hard_yellow_cap) else ("YELLOW" if final_score >= 30 else "RED")
    return {"confidence_score": final_score, "confidence_zone": zone, "confidence_reasons": reasons}


def compute_door_confidence(
    door_res: Mapping[str, Any],
    house_conf: Mapping[str, Any],
    prior: ParcelPrior,
    operating_point: str = CONFIDENCE_OPERATING_POINT,
) -> dict[str, Any]:
    """Use Case 2 (Conditional Front-Door Localization) score (0-100) and zone (`GREEN`/`YELLOW`/`OCCLUDED`/`SKIPPED`)."""
    house_zone = str(house_conf.get("confidence_zone") or "RED")
    d_status = str(door_res.get("door_status") or "SKIPPED_HOUSE_RED")
    if house_zone == "RED" or d_status == "SKIPPED_HOUSE_RED":
        return {"confidence_score": 0, "confidence_zone": "SKIPPED", "confidence_reasons": ["SKIPPED_HOUSE_NOT_CONFIRMED"]}
    if d_status == "DOOR_OCCLUDED":
        return {"confidence_score": 15, "confidence_zone": "OCCLUDED", "confidence_reasons": ["FRONT_DOOR_OCCLUDED_OR_NOT_VISIBLE_IN_SVI"]}
    if d_status == "DOOR_SINGLE_VIEW":
        return {"confidence_score": 32, "confidence_zone": "OCCLUDED", "confidence_reasons": ["SINGLE_DOOR_VIEW_ONLY_NOT_3D_TRIANGULABLE"]}

    reasons: list[str] = []
    n_door = int(door_res.get("n_door_views") or 0)
    n_porch = int(door_res.get("n_porch_views") or 0)
    d_rms = door_res.get("door_ray_rms_m")
    d_par = float(door_res.get("door_parallax_deg") or 0.0)

    if d_status == "PORCH_FALLBACK":
        sc = 48.0 + (8.0 if n_porch >= 3 else 3.0) + (6.0 if (d_rms is not None and float(d_rms) <= 0.75) else 0.0)
        reasons.append(f"PORCH_FALLBACK_NO_DIRECT_DOOR({n_porch}_VIEWS)")
        return {"confidence_score": int(round(min(64.0, sc))), "confidence_zone": "YELLOW", "confidence_reasons": reasons}

    score = 45.0 if n_door >= 4 else (38.0 if n_door == 3 else 24.0)
    reasons.append(f"PURE_DOOR_LOCK_{n_door}_VIEWS" if n_door >= 3 else "PURE_DOOR_2RAY_LOCK")
    if d_rms is not None:
        rf = float(d_rms)
        score += 28.0 if rf <= 0.40 else (22.0 if rf <= 0.85 else (14.0 if rf <= 1.25 else (7.0 if rf <= MAX_TIER_A_RMS_M else 0.0)))
    score += 15.0 if d_par >= 30.0 else (11.0 if d_par >= 18.0 else (6.0 if d_par >= 12.0 else 0.0))
    score += 12.0 if house_zone == "GREEN" else 5.0

    hard_yellow = False
    if d_status != "DOOR_LOCKED_3PLUS":
        hard_yellow = True
        reasons.append("FEWER_THAN_3_PURE_DOOR_RAYS_CAP")
    if d_rms is None or float(d_rms) > 1.25:
        hard_yellow = True
        reasons.append("DOOR_RMS_ABOVE_1.25M_CAP")
    if d_par < 12.0 and operating_point != "high_recall":
        hard_yellow = True
        reasons.append("LOW_DOOR_PARALLAX_CAP")
    if house_zone != "GREEN" and operating_point != "high_recall":
        hard_yellow = True
        reasons.append("HOUSE_ZONE_YELLOW_INHERITED")
    if prior.is_multi_unit and operating_point == "strict":
        hard_yellow = True
        reasons.append("MULTI_UNIT_MULTI_DOOR_AMBIGUITY_CAP")

    green_thresh = 80 if operating_point == "strict" else (60 if operating_point == "high_recall" else 70)
    if operating_point == "strict" and (d_rms is None or float(d_rms) > 0.65 or n_door < 3):
        hard_yellow = True
        reasons.append("STRICT_DOOR_REQUIRES_3PLUS_RAYS_LE_0.65M")

    final_score = int(round(max(20.0, min(100.0, score))))
    if hard_yellow and final_score >= green_thresh:
        final_score = green_thresh - 2
    zone = "GREEN" if (final_score >= green_thresh and not hard_yellow) else "YELLOW"
    return {"confidence_score": final_score, "confidence_zone": zone, "confidence_reasons": reasons}


def _solve_pure_door_stage(
    prior: ParcelPrior, all_views: Sequence[TargetRayView], house_detected: bool
) -> dict[str, Any]:
    """Use Case 2: Pure `door`-ray solver (ZERO `facade_center` rays)."""
    base_empty = {
        "door_status": "SKIPPED_HOUSE_RED", "door_method": "NONE",
        "door_lat": None, "door_lng": None, "persistable_door": False,
        "door_display_ephemeral_lat": None, "door_display_ephemeral_lng": None,
        "n_door_views": 0, "n_porch_views": 0,
        "door_ray_rms_m": None, "door_parallax_deg": 0.0, "door_views_used": [],
    }
    if not house_detected or not all_views:
        return base_empty

    door_by_pano: dict[str, TargetRayView] = {}
    porch_by_pano: dict[str, TargetRayView] = {}
    for v in sorted(all_views, key=lambda x: -x.weight):
        if v.keypoint_type == "door":
            door_by_pano.setdefault(v.pano_id, v)
        elif v.keypoint_type == "porch":
            porch_by_pano.setdefault(v.pano_id, v)

    d_views, p_views = list(door_by_pano.values()), list(porch_by_pano.values())
    n_d_total, n_p_total = len(d_views), len(p_views)

    if n_d_total >= 2:
        chosen_d, pt_d, rms_d, par_d, valid_d = _cauchy_ransac_and_solve(prior, d_views, [v.weight for v in d_views], gate_m=2.0)
        k_d = len(chosen_d)
        if pt_d is not None and valid_d:
            d_lat, d_lng = en_to_lla(float(pt_d[0]), float(pt_d[1]), prior.ref_lat, prior.ref_lng)
            eph_clamp = _clamp_point_to_polygon_ephemeral(prior, pt_d, max_shift_m=3.5)
            if k_d >= 3 and par_d >= MIN_TIER_A_PARALLAX_DEG and rms_d <= MAX_TIER_A_RMS_M:
                return {
                    "door_status": "DOOR_LOCKED_3PLUS", "door_method": "SV_PANO_PURE_DOOR_MULTIRAY",
                    "door_lat": round(d_lat, 7), "door_lng": round(d_lng, 7), "persistable_door": True,
                    "door_display_ephemeral_lat": eph_clamp[0] if eph_clamp else None,
                    "door_display_ephemeral_lng": eph_clamp[1] if eph_clamp else None,
                    "n_door_views": k_d, "n_porch_views": n_p_total,
                    "door_ray_rms_m": round(rms_d, 3), "door_parallax_deg": round(par_d, 1), "door_views_used": chosen_d,
                }
            if k_d >= 2 and par_d >= 8.0 and rms_d <= 2.2:
                return {
                    "door_status": "DOOR_LOCKED_2RAY", "door_method": "SV_PANO_PURE_DOOR_2RAY",
                    "door_lat": round(d_lat, 7), "door_lng": round(d_lng, 7), "persistable_door": True,
                    "door_display_ephemeral_lat": eph_clamp[0] if eph_clamp else None,
                    "door_display_ephemeral_lng": eph_clamp[1] if eph_clamp else None,
                    "n_door_views": k_d, "n_porch_views": n_p_total,
                    "door_ray_rms_m": round(rms_d, 3), "door_parallax_deg": round(par_d, 1), "door_views_used": chosen_d,
                }

    if n_d_total >= 1:
        v0 = d_views[0]
        dv0 = np.array([math.sin(math.radians(v0.az_key_deg)), math.cos(math.radians(v0.az_key_deg))], dtype=np.float64)
        hit_en = _intersect_ray_with_prior_ephemeral(prior, v0.cam_en, dv0)
        eph_ll = en_to_lla(float(hit_en[0]), float(hit_en[1]), prior.ref_lat, prior.ref_lng) if hit_en is not None else None
        return {
            "door_status": "DOOR_SINGLE_VIEW", "door_method": "SINGLE_DOOR_RAY_EPHEMERAL_ONLY",
            "door_lat": None, "door_lng": None, "persistable_door": False,
            "door_display_ephemeral_lat": round(eph_ll[0], 7) if eph_ll else None,
            "door_display_ephemeral_lng": round(eph_ll[1], 7) if eph_ll else None,
            "n_door_views": n_d_total, "n_porch_views": n_p_total,
            "door_ray_rms_m": None, "door_parallax_deg": 0.0, "door_views_used": d_views[:1],
        }

    if n_p_total >= 2:
        chosen_p, pt_p, rms_p, par_p, valid_p = _cauchy_ransac_and_solve(prior, p_views, [v.weight for v in p_views], gate_m=2.4)
        if pt_p is not None and valid_p and len(chosen_p) >= 2 and par_p >= 8.0 and rms_p <= 2.2:
            p_lat, p_lng = en_to_lla(float(pt_p[0]), float(pt_p[1]), prior.ref_lat, prior.ref_lng)
            eph_clamp = _clamp_point_to_polygon_ephemeral(prior, pt_p, max_shift_m=3.5)
            return {
                "door_status": "PORCH_FALLBACK", "door_method": "SV_PANO_PURE_PORCH_MULTIRAY",
                "door_lat": round(p_lat, 7), "door_lng": round(p_lng, 7), "persistable_door": True,
                "door_display_ephemeral_lat": eph_clamp[0] if eph_clamp else None,
                "door_display_ephemeral_lng": eph_clamp[1] if eph_clamp else None,
                "n_door_views": 0, "n_porch_views": len(chosen_p),
                "door_ray_rms_m": round(rms_p, 3), "door_parallax_deg": round(par_p, 1), "door_views_used": chosen_p,
            }

    return {**base_empty, "door_status": "DOOR_OCCLUDED", "n_porch_views": n_p_total}


def solve_house_and_door_decoupled(
    prior: ParcelPrior,
    views: Sequence[TargetRayView],
    raw_detections: Sequence[Mapping[str, Any]] | None = None,
    ocr_meta: Mapping[str, Any] | None = None,
    operating_point: str = CONFIDENCE_OPERATING_POINT,
) -> tuple[dict[str, Any], dict[str, Any]]:
    """Runs decoupled Stage 4A (Use Case 1: House Detection) and Stage 4B (Use Case 2: Conditional Front-Door)."""
    empty_house = {
        "status": prior.stage1_status if prior.stage1_status != "OK" else "MISSED",
        "tier": "REJECTED" if prior.stage1_status != "OK" else "MISSED",
        "method": "STAGE1_GATE" if prior.stage1_status != "OK" else "NONE",
        "stage1_reason": prior.stage1_reason, "privacy_blurred": False, "blurred_frame_count": 0,
        "lidar_used": False, "lidar_scans_fetched": 0, "lidar_plane_rms_m": None,
        "n_positions": 0, "n_images": 0, "n_door_views": 0, "n_porch_views": 0,
        "house_lat": None, "house_lng": None, "persistable_house": False,
        "building_centroid_ephemeral_lat": None, "building_centroid_ephemeral_lng": None,
        "ray_rms_m": None, "max_parallax_deg": 0.0, "ocr_verification_status": "UNREAD",
        "verified_house_number": None, "observed_ocr_numbers": [], "ocr_conflict_count": 0,
        "features": {}, "feature_agreement": {}, "views_used": [],
    }
    if prior.stage1_status != "OK":
        h_conf = compute_house_confidence(empty_house, prior, operating_point=operating_point)
        d_raw = _solve_pure_door_stage(prior, [], house_detected=False)
        return {**empty_house, **h_conf}, {**d_raw, **compute_door_confidence(d_raw, h_conf, prior, operating_point=operating_point)}

    raw_list = list(raw_detections or [])
    conflict_cnt = int((ocr_meta or {}).get("conflict_count", 0))
    conflict_nums = list((ocr_meta or {}).get("conflict_numbers", []))
    is_blur, n_blur = _is_privacy_blurred_t4_safe(prior, raw_list, views)
    if is_blur:
        blur_h = {
            **empty_house, "status": "PRIVACY_BLURRED_IN_SVI", "tier": "PRIVACY_BLURRED",
            "method": "NONE", "stage1_reason": None, "privacy_blurred": True,
            "blurred_frame_count": n_blur, "ocr_conflict_count": conflict_cnt,
        }
        h_conf = compute_house_confidence(blur_h, prior, operating_point=operating_point)
        d_raw = _solve_pure_door_stage(prior, [], house_detected=False)
        return {**blur_h, **h_conf}, {**d_raw, **compute_door_confidence(d_raw, h_conf, prior, operating_point=operating_point)}

    if not views:
        miss_h = {
            **empty_house,
            "status": "OCR_CONFLICT_REJECTED" if conflict_cnt > 0 else "MISSED",
            "tier": "MISSED", "method": "NONE", "stage1_reason": None, "blurred_frame_count": n_blur,
            "ocr_verification_status": "CONFLICT" if conflict_cnt > 0 else "UNREAD",
            "observed_ocr_numbers": conflict_nums, "ocr_conflict_count": conflict_cnt,
        }
        h_conf = compute_house_confidence(miss_h, prior, operating_point=operating_point)
        d_raw = _solve_pure_door_stage(prior, [], house_detected=False)
        return {**miss_h, **h_conf}, {**d_raw, **compute_door_confidence(d_raw, h_conf, prior, operating_point=operating_point)}

    # Stage 4A — House Detection across all building rays via Cauchy RANSAC
    by_pano: dict[str, TargetRayView] = {}
    for v in sorted(views, key=lambda x: -x.weight):
        by_pano.setdefault(v.pano_id, v)
    uviews = list(by_pano.values())
    n_door_tot = sum(v.keypoint_type == "door" for v in uviews)
    n_porch_tot = sum(v.keypoint_type == "porch" for v in uviews)
    weights_h = [
        v.weight * (1.35 if v.keypoint_type == "door" else (0.28 if (v.keypoint_type == "facade_center" and (n_door_tot >= 2 or (n_door_tot + n_porch_tot) >= 3)) else 1.0))
        for v in uviews
    ]
    chosen, pt_est, rms_m, max_par, valid_front = _cauchy_ransac_and_solve(
        prior, uviews, weights_h, gate_m=(2.2 if n_door_tot >= 2 else 2.6)
    )
    k_cnt = len(chosen)

    obs_nums = [v.ocr_number for v in chosen if v.ocr_number] + conflict_nums
    exact_views = [v.ocr_number for v in chosen if v.ocr_match_type == "EXACT_MATCH" and v.ocr_number]
    sib_views = [v.ocr_number for v in chosen if v.ocr_match_type == "SIBLING_UNIT" and v.ocr_number]
    if exact_views:
        ocr_stat, verified_num = "EXACT_MATCH", max(set(exact_views), key=exact_views.count)
    elif sib_views:
        ocr_stat, verified_num = "SIBLING_UNIT", None
    elif conflict_cnt >= 2 or (conflict_cnt >= 1 and conflict_cnt >= k_cnt):
        ocr_stat, verified_num = "CONFLICT", None
    else:
        ocr_stat, verified_num = "UNREAD", None

    feats, agree = _aggregate_logistics_features(chosen)
    n_doors_chosen = sum(v.keypoint_type == "door" for v in chosen)
    n_porches_chosen = sum(v.keypoint_type == "porch" for v in chosen)

    if pt_est is not None and k_cnt >= 3 and valid_front and max_par >= MIN_TIER_A_PARALLAX_DEG and rms_m <= MAX_TIER_A_RMS_M:
        h_lat, h_lng = en_to_lla(float(pt_est[0]), float(pt_est[1]), prior.ref_lat, prior.ref_lng)
        c_lat, c_lng = _ephemeral_building_centroid(prior, pt_est, prior.inward_normal_en)
        house_res = {
            **empty_house, "status": "TIER_A_RGB", "tier": "TIER_A", "method": "SV_PANO_PURE_MULTIRAY",
            "stage1_reason": None, "blurred_frame_count": n_blur, "n_positions": k_cnt, "n_images": k_cnt,
            "n_door_views": n_doors_chosen, "n_porch_views": n_porches_chosen,
            "house_lat": round(h_lat, 7), "house_lng": round(h_lng, 7), "persistable_house": True,
            "building_centroid_ephemeral_lat": c_lat, "building_centroid_ephemeral_lng": c_lng,
            "ray_rms_m": round(rms_m, 3), "max_parallax_deg": round(max_par, 1),
            "ocr_verification_status": ocr_stat, "verified_house_number": verified_num,
            "observed_ocr_numbers": obs_nums, "ocr_conflict_count": conflict_cnt,
            "features": feats, "feature_agreement": agree, "views_used": chosen,
        }
    elif pt_est is not None and k_cnt >= 2 and valid_front and max_par >= 8.0 and rms_m <= 2.2:
        h_lat, h_lng = en_to_lla(float(pt_est[0]), float(pt_est[1]), prior.ref_lat, prior.ref_lng)
        c_lat, c_lng = _ephemeral_building_centroid(prior, pt_est, prior.inward_normal_en)
        house_res = {
            **empty_house, "status": "TIER_B_RGB", "tier": "TIER_B", "method": "SV_PANO_PURE_2RAY",
            "stage1_reason": None, "blurred_frame_count": n_blur, "n_positions": k_cnt, "n_images": k_cnt,
            "n_door_views": n_doors_chosen, "n_porch_views": n_porches_chosen,
            "house_lat": round(h_lat, 7), "house_lng": round(h_lng, 7), "persistable_house": True,
            "building_centroid_ephemeral_lat": c_lat, "building_centroid_ephemeral_lng": c_lng,
            "ray_rms_m": round(rms_m, 3), "max_parallax_deg": round(max_par, 1),
            "ocr_verification_status": ocr_stat, "verified_house_number": verified_num,
            "observed_ocr_numbers": obs_nums, "ocr_conflict_count": conflict_cnt,
            "features": feats, "feature_agreement": agree, "views_used": chosen,
        }
    else:
        hit_pts, hit_w = [], []
        for v in chosen:
            dv = np.array([math.sin(math.radians(v.az_key_deg)), math.cos(math.radians(v.az_key_deg))], dtype=np.float64)
            hp = _intersect_ray_with_prior_ephemeral(prior, v.cam_en, dv)
            if hp is not None:
                hit_pts.append(hp)
                hit_w.append(v.weight)
        h_en = np.average(np.array(hit_pts), axis=0, weights=np.array(hit_w)) if hit_pts else (chosen[0].cam_en + 15.0 * np.array([math.sin(math.radians(chosen[0].az_key_deg)), math.cos(math.radians(chosen[0].az_key_deg))]))
        h_lat, h_lng = en_to_lla(float(h_en[0]), float(h_en[1]), prior.ref_lat, prior.ref_lng)
        c_lat, c_lng = _ephemeral_building_centroid(prior, h_en, prior.inward_normal_en)
        house_res = {
            **empty_house, "status": "TIER_B_RGB", "tier": "TIER_B", "method": "SV_PANO_RAY_PRIOR_EPHEMERAL",
            "stage1_reason": None, "blurred_frame_count": n_blur, "n_positions": k_cnt, "n_images": k_cnt,
            "n_door_views": n_doors_chosen, "n_porch_views": n_porches_chosen,
            "house_lat": round(h_lat, 7), "house_lng": round(h_lng, 7), "persistable_house": False,
            "building_centroid_ephemeral_lat": c_lat, "building_centroid_ephemeral_lng": c_lng,
            "ray_rms_m": None, "max_parallax_deg": round(max_par, 1),
            "ocr_verification_status": ocr_stat, "verified_house_number": verified_num,
            "observed_ocr_numbers": obs_nums, "ocr_conflict_count": conflict_cnt,
            "features": feats, "feature_agreement": agree, "views_used": chosen,
        }

    h_conf = compute_house_confidence(house_res, prior, operating_point=operating_point)
    house_out = {**house_res, **h_conf}
    bal_house_detected = compute_house_confidence(house_res, prior, operating_point="balanced")["confidence_zone"] != "RED"
    door_raw = _solve_pure_door_stage(prior, views, house_detected=bal_house_detected)
    door_out = {**door_raw, **compute_door_confidence(door_raw, h_conf, prior, operating_point=operating_point)}
    return house_out, door_out


def _lla_to_ecef(lat_deg: float | np.ndarray, lon_deg: float | np.ndarray, alt_m: float | np.ndarray) -> np.ndarray:
    lat_r, lon_r, alt = np.radians(np.asarray(lat_deg, dtype=np.float64)), np.radians(np.asarray(lon_deg, dtype=np.float64)), np.asarray(alt_m, dtype=np.float64)
    slat, clat = np.sin(lat_r), np.cos(lat_r)
    n_val = WGS84_A / np.sqrt(1.0 - WGS84_E2 * (slat * slat))
    x, y, z = (n_val + alt) * clat * np.cos(lon_r), (n_val + alt) * clat * np.sin(lon_r), (n_val * (1.0 - WGS84_E2) + alt) * slat
    return np.array([float(x), float(y), float(z)], dtype=np.float64) if np.ndim(x) == 0 else np.column_stack([x, y, z]).astype(np.float64, copy=False)


def _ecef_to_enu(pts_ecef: np.ndarray, ref_lat: float, ref_lng: float, ref_alt: float) -> np.ndarray:
    origin = _lla_to_ecef(ref_lat, ref_lng, ref_alt)
    lat_r, lon_r = math.radians(ref_lat), math.radians(ref_lng)
    slat, clat, slon, clon = math.sin(lat_r), math.cos(lat_r), math.sin(lon_r), math.cos(lon_r)
    r_mat = np.array([[-slon, clon, 0.0], [-slat * clon, -slat * slon, clat], [clat * clon, clat * slon, slat]], dtype=np.float64)
    return ((np.asarray(pts_ecef, dtype=np.float64) - origin) @ r_mat.T).astype(np.float32)


def fetch_pano_lidar_batch_ephemeral(
    priors_by_idx: Mapping[int, ParcelPrior], radius_m: float = 36.0, verbose: bool = True
) -> dict[int, pd.DataFrame]:
    """Joins `pano_observations_latest` + `pano_lidar_metadata_latest` across candidate stops via `UNNEST(@batch_pts)`."""
    out_map: dict[int, pd.DataFrame] = {idx: pd.DataFrame() for idx in priors_by_idx}
    struct_vals: list[bigquery.StructQueryParameter] = []
    for idx, p in priors_by_idx.items():
        if p.stage1_status != "OK":
            continue
        fm_lat, fm_lng = en_to_lla(float(p.front_mid_en[0]), float(p.front_mid_en[1]), p.ref_lat, p.ref_lng)
        struct_vals.append(bigquery.StructQueryParameter(
            None,
            bigquery.ScalarQueryParameter("idx", "INT64", int(idx)),
            bigquery.ScalarQueryParameter("lat", "FLOAT64", float(fm_lat)),
            bigquery.ScalarQueryParameter("lng", "FLOAT64", float(fm_lng)),
        ))
    if not struct_vals:
        return out_map

    sql = f"""
    WITH stops AS (
      SELECT s.idx, ST_GEOGPOINT(s.lng, s.lat) AS g_pt
      FROM UNNEST(@batch_pts) AS s
    )
    SELECT
      s.idx AS stop_idx,
      o.observation_id,
      o.capture_id,
      o.camera_pose.latitude AS cam_lat,
      o.camera_pose.longitude AS cam_lng,
      o.camera_pose.altitude AS cam_alt,
      o.camera_pose.heading AS heading,
      l.device_number,
      l.uri AS laz_uri,
      l.count_points
    FROM stops s
    JOIN `{SVI_PROJECT_ID}.{SVI_DATASET}.pano_observations_latest` o
      ON ST_DWITHIN(ST_GEOGPOINT(o.camera_pose.longitude, o.camera_pose.latitude), s.g_pt, @radius_m)
    JOIN `{SVI_PROJECT_ID}.{SVI_LIDAR_DATASET}.pano_lidar_metadata_latest` l
      ON o.observation_id = l.observation_id
    WHERE l.count_points >= 8000
    """
    struct_type = bigquery.StructQueryParameterType(
        bigquery.ScalarQueryParameterType("INT64", name="idx"),
        bigquery.ScalarQueryParameterType("FLOAT64", name="lat"),
        bigquery.ScalarQueryParameterType("FLOAT64", name="lng"),
    )
    params = [
        bigquery.ArrayQueryParameter("batch_pts", struct_type, struct_vals),
        bigquery.ScalarQueryParameter("radius_m", "FLOAT64", float(radius_m)),
    ]
    try:
        dry_cfg = bigquery.QueryJobConfig(query_parameters=params, dry_run=True, use_query_cache=False)
        est_gb = bq_client.query(sql, job_config=dry_cfg).total_bytes_processed / 1e9
        if est_gb > MAX_QUERY_GB:
            logger.warning("Skipping LiDAR query: estimated %.2f GB exceeds MAX_QUERY_GB=%.1f", est_gb, MAX_QUERY_GB)
            return out_map
        if verbose:
            print(f"BigQuery pano_lidar_metadata_latest dry-run ({len(struct_vals)} stop(s)): {est_gb:.2f} GB")
        run_cfg = bigquery.QueryJobConfig(query_parameters=params, maximum_bytes_billed=int(MAX_QUERY_GB * 1e9))
        df = bq_client.query(sql, job_config=run_cfg).to_dataframe()
    except GoogleAPICallError as exc:
        logger.warning("pano_lidar_metadata_latest query failed or dataset not subscribed: %s", exc)
        return out_map

    if not df.empty:
        for stop_idx, grp in df.groupby("stop_idx"):
            out_map[int(stop_idx)] = grp.drop(columns=["stop_idx"]).reset_index(drop=True)
    return out_map


def surgical_lidar_wall_lock(
    prior: ParcelPrior,
    house_result: dict[str, Any],
    door_result: dict[str, Any],
    lidar_candidates_df: pd.DataFrame | None = None,
    max_laz_files: int = 2,
    operating_point: str = CONFIDENCE_OPERATING_POINT,
) -> tuple[dict[str, Any], dict[str, Any]]:
    """Hardened Stage 5 3D LiDAR verification: >=2 rays, >=12 deg parallax, <=1.0 m agreement, <=25 deg normal, farthest slab."""
    if not ENABLE_SURGICAL_LIDAR_FALLBACK or prior.stage1_status != "OK" or house_result.get("tier") == "TIER_A" or house_result.get("privacy_blurred"):
        return house_result, door_result

    views: list[TargetRayView] = house_result.get("views_used") or []
    by_pano: dict[str, TargetRayView] = {}
    for v in sorted(views, key=lambda x: -x.weight):
        by_pano.setdefault(v.pano_id, v)
    uviews = list(by_pano.values())
    if len(uviews) < 2:
        return house_result, door_result

    pre_par = max((angle_between_rays_deg(uviews[a].az_key_deg, uviews[b].az_key_deg) for a in range(len(uviews)) for b in range(a + 1, len(uviews))), default=0.0)
    if pre_par < 12.0:
        return house_result, door_result

    cand_df = lidar_candidates_df if lidar_candidates_df is not None else fetch_pano_lidar_batch_ephemeral({0: prior}, verbose=True)[0]
    if cand_df.empty:
        return house_result, door_result

    fm_lat, fm_lng = en_to_lla(float(prior.front_mid_en[0]), float(prior.front_mid_en[1]), prior.ref_lat, prior.ref_lng)
    m_rad, n_cos = _wgs84_radii(fm_lat)
    dy, dx = np.radians(cand_df["cam_lat"] - fm_lat) * m_rad, np.radians(cand_df["cam_lng"] - fm_lng) * n_cos
    dist_m = np.hypot(dx, dy)
    near_l = cand_df[(dist_m >= 3.5) & (dist_m <= 36.0)].copy()
    if near_l.empty:
        return house_result, door_result

    az_to_house = np.degrees(np.arctan2(-dx.loc[near_l.index], -dy.loc[near_l.index])) % 360.0
    hdg_diff = np.abs(((near_l["heading"].to_numpy() - az_to_house.to_numpy() + 180.0) % 360.0) - 180.0)
    near_l = near_l.assign(dist_house_m=dist_m.loc[near_l.index], hdg_diff_deg=hdg_diff)
    near_l = near_l[near_l["hdg_diff_deg"] <= 52.0].sort_values(["hdg_diff_deg", "dist_house_m", "count_points"], ascending=[True, True, False])
    if near_l.empty:
        return house_result, door_result

    top_obs = near_l.iloc[0]["observation_id"]
    chosen_laz = pd.concat([near_l[near_l["observation_id"] == top_obs], near_l[near_l["observation_id"] != top_obs]], ignore_index=True).head(max_laz_files)
    ref_alt = float(chosen_laz.iloc[0]["cam_alt"]) - CAM_HEIGHT_ABOVE_GROUND_M

    chunks, laz_fetched = [], 0
    for r in chosen_laz.itertuples(index=False):
        try:
            b, o = str(r.laz_uri).removeprefix("gs://").split("/", 1)
            resp_l = authed_http.get(f"https://storage.googleapis.com/storage/v1/b/{b}/o/{urllib.parse.quote(o, safe='')}?alt=media", timeout=(10, 60))
            if resp_l.status_code != 200:
                continue
            las = laspy.read(io.BytesIO(resp_l.content))
            pts_enu = _ecef_to_enu(np.column_stack([np.asarray(las.x, dtype=np.float64), np.asarray(las.y, dtype=np.float64), np.asarray(las.z, dtype=np.float64)]), prior.ref_lat, prior.ref_lng, ref_alt)
            keep = (np.abs(pts_enu[:, 0]) <= 45.0) & (np.abs(pts_enu[:, 1]) <= 45.0) & (pts_enu[:, 2] >= -15.0) & (pts_enu[:, 2] <= 20.0)
            chunks.append(pts_enu[keep])
            laz_fetched += 1
        except Exception as exc:
            logger.warning("LAZ stream/parse failed for %s: %s", r.laz_uri, exc)

    if not chunks:
        return house_result, door_result
    pts = np.vstack(chunks)
    if len(pts) < 80:
        return house_result, door_result

    rel_front = np.hypot(pts[:, 0] - float(prior.front_mid_en[0]), pts[:, 1] - float(prior.front_mid_en[1]))
    near_g = pts[rel_front <= 18.0, 2]
    z_ground = float(np.percentile(near_g if len(near_g) >= 30 else pts[:, 2], 10))
    z_agl = pts[:, 2] - z_ground

    in_n = prior.inward_normal_en
    lat_u = np.array([in_n[1], -in_n[0]], dtype=np.float64)
    rel_xy = pts[:, :2].astype(np.float64) - prior.front_mid_en
    d_in, d_lat = rel_xy @ in_n, rel_xy @ lat_u
    half_w = max(0.58 * prior.facade_width_m, 4.5)
    min_lat_span = max(2.5, 0.35 * prior.facade_width_m)
    mask = (z_agl >= 1.2) & (z_agl <= 4.6) & (d_in >= -4.0) & (d_in <= 4.0) & (np.abs(d_lat) <= half_w)
    if int(mask.sum()) < 45:
        return house_result, door_result

    d_in_w, d_lat_w, z_w, wall_pts = d_in[mask], d_lat[mask], z_agl[mask], pts[mask]
    qualifying: list[dict[str, Any]] = []
    cos_25 = math.cos(math.radians(25.0))
    for b0 in np.arange(-4.0, 3.8, 0.25):
        ms = (d_in_w >= b0) & (d_in_w <= b0 + 0.45)
        if int(ms.sum()) < 35:
            continue
        if float(np.percentile(z_w[ms], 92) - np.percentile(z_w[ms], 8)) < 1.50:
            continue
        if float(np.percentile(d_lat_w[ms], 95) - np.percentile(d_lat_w[ms], 5)) < min_lat_span:
            continue
        med_d = float(np.median(d_in_w[ms]))
        inl_xy = wall_pts[np.abs(d_in_w - med_d) <= 0.25, :2].astype(np.float64)
        if len(inl_xy) < 30:
            continue
        mean_w = inl_xy.mean(axis=0)
        _, _, vt = np.linalg.svd(inl_xy - mean_w, full_matrices=False)
        n_w = vt[-1] if float(np.dot(vt[-1], in_n)) >= 0.0 else -vt[-1]
        if float(np.dot(n_w, in_n)) < cos_25:
            continue
        plane_rms = float(np.sqrt(np.mean(((inl_xy - mean_w) @ n_w) ** 2)))
        if plane_rms <= 0.14:
            qualifying.append({"med_d": med_d, "cnt": len(inl_xy), "inlier_xy": inl_xy, "mean_wall": mean_w, "n_wall": n_w, "plane_rms": plane_rms})

    if not qualifying:
        return house_result, door_result
    max_cnt = max(s["cnt"] for s in qualifying)
    chosen_slab = max((s for s in qualifying if s["cnt"] >= 0.35 * max_cnt), key=lambda s: s["med_d"])
    inlier_xy, mean_wall, n_wall, wall_plane_rms = chosen_slab["inlier_xy"], chosen_slab["mean_wall"], chosen_slab["n_wall"], chosen_slab["plane_rms"]

    tan_wall = np.array([n_wall[1], -n_wall[0]], dtype=np.float64)
    lat_pts = (inlier_xy - mean_wall) @ tan_wall
    lat_lo, lat_hi = float(lat_pts.min()) - 1.8, float(lat_pts.max()) + 1.8

    ray_hits = []
    for v in uviews:
        dv = np.array([math.sin(math.radians(v.az_key_deg)), math.cos(math.radians(v.az_key_deg))], dtype=np.float64)
        nv = np.array([math.cos(math.radians(v.az_key_deg)), -math.sin(math.radians(v.az_key_deg))], dtype=np.float64)
        denom = float(np.dot(dv, n_wall))
        if abs(denom) < 0.20:
            continue
        t_hit = float(np.dot(mean_wall - v.cam_en, n_wall)) / denom
        if MIN_RAY_RANGE_M <= t_hit <= 48.0:
            hp = v.cam_en + t_hit * dv
            if lat_lo <= float((hp - mean_wall) @ tan_wall) <= lat_hi:
                ray_hits.append((v, hp, nv))

    best_sub = []
    for rh0 in ray_hits:
        sub = [rh for rh in ray_hits if float(np.linalg.norm(rh[1] - rh0[1])) <= 1.0]
        if len(sub) > len(best_sub):
            best_sub = sub
    if len(best_sub) < 2:
        return house_result, door_result

    hit_par = max((angle_between_rays_deg(best_sub[a][0].az_key_deg, best_sub[b][0].az_key_deg) for a in range(len(best_sub)) for b in range(a + 1, len(best_sub))), default=0.0)
    if hit_par < 12.0:
        return house_result, door_result

    sub_pts = np.array([rh[1] for rh in best_sub], dtype=np.float64)
    sub_w = np.array([rh[0].weight for rh in best_sub], dtype=np.float64)
    h_en = np.average(sub_pts, axis=0, weights=sub_w)
    if float(np.max(np.linalg.norm(sub_pts - h_en, axis=1))) > 1.0:
        return house_result, door_result

    perp_res = np.array([abs(float((h_en - rh[0].cam_en) @ rh[2])) for rh in best_sub], dtype=np.float64)
    ray_rms_on_plane = float(np.sqrt(np.average(perp_res * perp_res, weights=sub_w)))
    if ray_rms_on_plane > 0.85:
        return house_result, door_result

    h_lat, h_lng = en_to_lla(float(h_en[0]), float(h_en[1]), prior.ref_lat, prior.ref_lng)
    c_lat, c_lng = _ephemeral_building_centroid(prior, h_en, n_wall)
    used_views = [rh[0] for rh in best_sub]
    out_h = dict(house_result)
    out_h.update({
        "status": "TIER_A_LIDAR_ASSISTED", "tier": "TIER_A", "method": "SV_PANO_PLUS_SURGICAL_3D_LIDAR",
        "lidar_used": True, "lidar_scans_fetched": laz_fetched, "lidar_wall_points": int(len(inlier_xy)),
        "lidar_plane_rms_m": round(wall_plane_rms, 3),
        "n_positions": len(used_views), "n_images": len(used_views),
        "n_door_views": sum(v.keypoint_type == "door" for v in used_views),
        "n_porch_views": sum(v.keypoint_type == "porch" for v in used_views),
        "house_lat": round(h_lat, 7), "house_lng": round(h_lng, 7), "persistable_house": True,
        "building_centroid_ephemeral_lat": c_lat, "building_centroid_ephemeral_lng": c_lng,
        "ray_rms_m": round(ray_rms_on_plane, 3), "max_parallax_deg": round(hit_par, 1), "views_used": used_views,
    })
    h_conf = compute_house_confidence(out_h, prior, operating_point=operating_point)
    out_h.update(h_conf)
    out_d = {**door_result, **compute_door_confidence(door_result, h_conf, prior, operating_point=operating_point)}
    return out_h, out_d


ray_views, ocr_audit = build_target_ray_views(prior, selected_frames_df, crop_detections)
rgb_house, rgb_door = solve_house_and_door_decoupled(prior, ray_views, raw_detections=crop_detections, ocr_meta=ocr_audit)
house_result, door_result = surgical_lidar_wall_lock(prior, rgb_house, rgb_door)

print(f"=== Decoupled Result for {prior.formatted_address} ===")
print(f"  [UC1: House] Zone={house_result['confidence_zone']} ({house_result['confidence_score']}/100) | {house_result['status']} ({house_result['tier']})")
print(f"               Pos={house_result['n_positions']} | Ray RMS={house_result['ray_rms_m']} m | LiDAR RMS={house_result['lidar_plane_rms_m']} m | Reasons={house_result['confidence_reasons']}")
print(f"               Pure SVI House Pin: ({house_result['house_lat']}, {house_result['house_lng']}) [persistable={house_result['persistable_house']}]")
print(f"  [UC2: Door ] Zone={door_result['confidence_zone']} ({door_result['confidence_score']}/100) | {door_result['door_status']} ({door_result['door_method']})")
print(f"               Door Views={door_result['n_door_views']} | Porch Views={door_result['n_porch_views']} | Door RMS={door_result['door_ray_rms_m']} m | Reasons={door_result['confidence_reasons']}")
print(f"               Pure SVI Door Pin : ({door_result['door_lat']}, {door_result['door_lng']}) [persistable={door_result['persistable_door']}]")
print(f"               Ephemeral Clamp   : ({door_result['door_display_ephemeral_lat']}, {door_result['door_display_ephemeral_lng']}) [0-day display only]")
print(f"  Unprimed OCR : {house_result['ocr_verification_status']} (verified={house_result['verified_house_number']}, observed={house_result['observed_ocr_numbers']})")
print(f"  Delivery Acc.: {house_result['features']}")

## 7. Visual QA: Zoom crops, keypoint-colored rays & decoupled House (`H`) + Door (`D`) map

* **Top row**: Up to three target-conditioned zoom crops showing `facade_box_2d` (cyan), `porch_or_entryway_box_2d` (amber), and `front_door_box_2d` (lime) with `"Imagery © Google"` attribution.
* **Bottom panel**: Local metric East-North-Up axes (`East (m)` / `North (m)`) displaying the ephemeral `displayPolygon`, street reference point, Street View car positions, **keypoint-colored sightlines** (emerald = `door`, amber = `porch`, cyan = `facade_center`), the **Pure SVI House Pin (`H`)**, and — when locked — the separate **Pure SVI Front-Door Pin (`D`)** with `"Google Maps | Imagery © Google"` attribution.

> **Basemap Compliance Warning (GMP ToS §3.2.3(e)):** This diagnostic plot renders on plain metric `matplotlib` axes without a third-party basemap. **Never** overlay `displayPolygon`, `navigationPoints`, or GMP-derived outlines on OpenStreetMap, Mapbox, Folium, or other non-Google basemaps — use the Google Maps JavaScript API or Photorealistic 3D Tiles when rendering on an interactive map.

In [ ]:
# @title Step 7: Plot matched zoom crops & decoupled House (H) + Front-Door (D) ENU ray map
views_to_plot: list[TargetRayView] = house_result.get("views_used") or []
if not views_to_plot:
    print(
        f"No renderable views for {prior.formatted_address} "
        f"(House={house_result['confidence_zone']} {house_result['status']}, Door={door_result['confidence_zone']})."
    )
else:
    show_views = views_to_plot[:3]
    fig = plt.figure(figsize=(15, 10), constrained_layout=True)
    gs = fig.add_gridspec(2, 3, height_ratios=[1.05, 1.15])

    for idx_v, vw in enumerate(show_views):
        ax = fig.add_subplot(gs[0, idx_v])
        crop_pil = Image.open(io.BytesIO(crop_images_mem[vw.observation_id]))
        cw_px, ch_px = crop_pil.size
        ax.imshow(crop_pil)
        det = vw.raw_det
        for box_key, color, lbl in [
            ("facade_box_2d", "#00BCD4", "Facade"),
            ("porch_or_entryway_box_2d", "#FFB300", "Porch"),
            ("front_door_box_2d", "#00E676", "Door"),
        ]:
            b2d = det.get(box_key)
            if b2d and len(b2d) == 4:
                y0, x0, y1, x1 = [float(z) for z in b2d]
                rx0, ry0 = (x0 / 1000.0) * cw_px, (y0 / 1000.0) * ch_px
                rw, rh = ((x1 - x0) / 1000.0) * cw_px, ((y1 - y0) / 1000.0) * ch_px
                ax.add_patch(mpatches.Rectangle((rx0, ry0), rw, rh, fill=False, edgecolor=color, linewidth=2.2, label=lbl))
        ax.set_title(f"cam_{vw.cam_k} | {vw.keypoint_type} | OCR={vw.ocr_number or '-'} ({vw.ocr_mounted_on})", fontsize=9.5)
        ax.text(
            0.98, 0.02, "Imagery © Google", transform=ax.transAxes, ha="right", va="bottom",
            fontsize=8, color="white", bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.6),
        )
        ax.axis("off")

    ax_map = fig.add_subplot(gs[1, :])
    if prior.poly_en is not None:
        px, py = prior.poly_en.exterior.xy
        ax_map.fill(px, py, facecolor="#E8F0FE", edgecolor="#1A73E8", linewidth=2.0, label="GMP displayPolygon (ephemeral)")
    ax_map.plot(
        [prior.front_left_en[0], prior.front_right_en[0]],
        [prior.front_left_en[1], prior.front_right_en[1]],
        color="#1A73E8", linewidth=3.5, label="Street-facing front wall",
    )
    nav_e, nav_n = lla_to_en(prior.nav_lat, prior.nav_lng, prior.ref_lat, prior.ref_lng)
    ax_map.scatter([nav_e], [nav_n], s=95, marker="s", color="#FBBC04", edgecolor="black", zorder=5, label=f"Street ref ({prior.street_ref_source})")

    ray_palette = {"door": "#137333", "porch": "#E37400", "facade_center": "#00838F"}
    seen_ray_lbls: set[str] = set()
    for vw in views_to_plot:
        ce, cn = float(vw.cam_en[0]), float(vw.cam_en[1])
        dv = np.array([math.sin(math.radians(vw.az_key_deg)), math.cos(math.radians(vw.az_key_deg))])
        rc = ray_palette.get(vw.keypoint_type, "#00ACC1")
        lbl = f"Sightline ({vw.keypoint_type})" if vw.keypoint_type not in seen_ray_lbls else None
        seen_ray_lbls.add(vw.keypoint_type)
        ax_map.scatter([ce], [cn], s=55, color="#444746", zorder=4)
        ax_map.plot([ce, ce + 34.0 * dv[0]], [cn, cn + 34.0 * dv[1]], linestyle="--", linewidth=1.4, color=rc, alpha=0.85, label=lbl)

    if house_result["house_lat"] is not None:
        he, hn = lla_to_en(house_result["house_lat"], house_result["house_lng"], prior.ref_lat, prior.ref_lng)
        ax_map.scatter([he], [hn], s=140, marker="o", color="#0B57D0", edgecolor="white", linewidth=1.6, zorder=6, label="UC1: Pure SVI House Pin (H)")
    if door_result["door_lat"] is not None:
        de, dn = lla_to_en(door_result["door_lat"], door_result["door_lng"], prior.ref_lat, prior.ref_lng)
        ax_map.scatter([de], [dn], s=155, marker="D", color="#137333", edgecolor="white", linewidth=1.4, zorder=7, label=f"UC2: Pure SVI Door Pin (D, {door_result['door_status']})")
    elif door_result["door_display_ephemeral_lat"] is not None:
        de, dn = lla_to_en(door_result["door_display_ephemeral_lat"], door_result["door_display_ephemeral_lng"], prior.ref_lat, prior.ref_lng)
        ax_map.scatter([de], [dn], s=110, marker="D", facecolor="none", edgecolor="#E37400", linewidth=2.0, zorder=7, label="UC2: Ephemeral Single-View Door Hint")
    if house_result["building_centroid_ephemeral_lat"] is not None:
        cen_e, cen_n = lla_to_en(house_result["building_centroid_ephemeral_lat"], house_result["building_centroid_ephemeral_lng"], prior.ref_lat, prior.ref_lng)
        ax_map.scatter([cen_e], [cen_n], s=75, marker="x", color="#B3261E", linewidth=2.0, zorder=6, label="Ephemeral Building Center (0-day)")

    ax_map.set_aspect("equal", adjustable="datalim")
    ax_map.grid(True, linestyle=":", alpha=0.5)
    ax_map.set_xlabel("East (m from parcel reference)")
    ax_map.set_ylabel("North (m from parcel reference)")
    ax_map.set_title(
        f"{prior.formatted_address}\n"
        f"UC1 House: {house_result['confidence_zone']} ({house_result['confidence_score']}/100, {house_result['status']}, RMS={house_result['ray_rms_m']} m)  |  "
        f"UC2 Door: {door_result['confidence_zone']} ({door_result['confidence_score']}/100, {door_result['door_status']}, RMS={door_result['door_ray_rms_m']} m)",
        fontsize=10.5,
    )
    ax_map.legend(loc="upper right", fontsize=8.0)
    ax_map.text(0.01, 0.02, "Google Maps | Imagery © Google", transform=ax_map.transAxes, fontsize=8.5, fontweight="bold", color="#3C4043")
    plt.show()

## 8. Batched ephemeral `UNNEST(@batch_pts)` helper across decoupled House & Door scenarios

`locate_houses_batch_ephemeral(addresses)` executes the full 5-stage decoupled pipeline across an entire stop list using **a single Float-BBox `pano_observations_latest` BigQuery scan** (plus at most one `pano_lidar_metadata_latest` scan for any non-Tier-A candidates), returning separate **Use Case 1 (`house_*`)** and **Use Case 2 (`door_*`)** columns across all canonical scenarios:
1. **House `GREEN` + Door `GREEN` (`530 W Myrtle St`)** — Clear single-family house with `>= 3` pure door rays (`DOOR_LOCKED_3PLUS`) and unprimed exact house-number OCR (`"530"`).
2. **House `GREEN` + Door `OCCLUDED` (`23 N Davis Ave`)** — Confirmed `TIER_A` house whose front door is only partially visible (`DOOR_SINGLE_VIEW` / `OCCLUDED`): Use Case 1 stays `GREEN` while Use Case 2 cleanly abstains (`door_lat=None`) instead of guessing.
3. **House `YELLOW` (`701 Lime St`)** — Multi-unit residential building (`4` `subpremise` unit children in `subDestinations`), automatically capped at `YELLOW` (`MULTI_UNIT_SHARED_STRUCTURE_CAP`).
4. **House `RED` + Door `SKIPPED` (`2521 W Parker St` & `PO Box 123`)** — Homeowner Street View opt-out (`PRIVACY_BLURRED_IN_SVI`, `0` attributes leaked) and Stage-1 non-physical postal box gate (`ENTITY_MISMATCH`, `$0` BigQuery/Gemini spend).

In [ ]:
# @title Step 8: Batched UNNEST(@batch_pts) helper & decoupled House vs. Door multi-address demo
def locate_houses_batch_ephemeral(addresses: Sequence[str], max_panos: int = MAX_PANOS) -> pd.DataFrame:
    """Resolves a batch of US addresses using ONE Float-BBox `UNNEST(@batch_pts)` BigQuery scan."""
    priors_init = [resolve_address_gmp(a) for a in addresses]
    pano_by_idx = fetch_panos_batch_ephemeral(priors_init, verbose=True)

    rgb_house_map: dict[int, dict[str, Any]] = {}
    rgb_door_map: dict[int, dict[str, Any]] = {}
    active_priors: dict[int, ParcelPrior] = {}
    frame_stats_map: dict[int, dict[str, Any]] = {}
    usage_map: dict[int, dict[str, Any]] = {}
    lidar_candidates_needed: dict[int, ParcelPrior] = {}

    for idx, p0 in enumerate(priors_init):
        if p0.stage1_status != "OK":
            active_priors[idx] = p0
            frame_stats_map[idx] = {"selected_frames": 0, "frame_reduction_x": 0.0}
            usage_map[idx] = {"est_online_cost_usd": 0.0, "est_batch_cost_usd": 0.0}
            h_sol, d_sol = solve_house_and_door_decoupled(p0, [])
            rgb_house_map[idx], rgb_door_map[idx] = h_sol, d_sol
            continue

        sel_df, f_stats, act_p = select_optimal_house_frames(p0, pano_by_idx[idx], max_panos=max_panos)
        dets, g_usage, _ = detect_target_house_crops(act_p, sel_df, keep_crop_bytes=False)
        r_views, ocr_meta = build_target_ray_views(act_p, sel_df, dets)
        h_sol, d_sol = solve_house_and_door_decoupled(act_p, r_views, raw_detections=dets, ocr_meta=ocr_meta)

        active_priors[idx] = act_p
        frame_stats_map[idx] = f_stats
        usage_map[idx] = g_usage
        rgb_house_map[idx], rgb_door_map[idx] = h_sol, d_sol
        if ENABLE_SURGICAL_LIDAR_FALLBACK and h_sol["tier"] != "TIER_A" and not h_sol["privacy_blurred"] and h_sol["n_positions"] >= 2:
            lidar_candidates_needed[idx] = act_p

    lidar_by_idx = (
        fetch_pano_lidar_batch_ephemeral(lidar_candidates_needed, verbose=True)
        if lidar_candidates_needed
        else {}
    )

    rows_out = []
    for idx, addr in enumerate(addresses):
        p = active_priors[idx]
        if idx in lidar_candidates_needed:
            final_h, final_d = surgical_lidar_wall_lock(
                p, rgb_house_map[idx], rgb_door_map[idx], lidar_candidates_df=lidar_by_idx.get(idx, pd.DataFrame())
            )
        else:
            final_h, final_d = rgb_house_map[idx], rgb_door_map[idx]
        rows_out.append({
            "query_address": addr,
            "house_zone": final_h["confidence_zone"],
            "house_score": final_h["confidence_score"],
            "house_status": final_h["status"],
            "house_lat": final_h["house_lat"],
            "house_lng": final_h["house_lng"],
            "persistable_house": final_h["persistable_house"],
            "door_zone": final_d["confidence_zone"],
            "door_score": final_d["confidence_score"],
            "door_status": final_d["door_status"],
            "door_lat": final_d["door_lat"],
            "door_lng": final_d["door_lng"],
            "persistable_door": final_d["persistable_door"],
            "door_rms_m": final_d["door_ray_rms_m"],
            "street_ref": p.street_ref_source,
            "is_multi_unit": p.is_multi_unit,
            "frames_sent": frame_stats_map[idx]["selected_frames"],
            "blurred_frames": final_h["blurred_frame_count"],
            "car_positions": final_h["n_positions"],
            "ocr_status": final_h["ocr_verification_status"],
            "verified_ocr": final_h["verified_house_number"],
            "house_rms_m": final_h["ray_rms_m"],
            "est_online_usd": usage_map[idx]["est_online_cost_usd"],
            "features": final_h["features"],
        })
    return pd.DataFrame(rows_out)


SAMPLE_US_ADDRESSES = [
    "530 W Myrtle St, Lakeland, FL 33815",
    "23 N Davis Ave, Lakeland, FL 33805",
    "701 Lime St, Lakeland, FL 33801",
    "2521 W Parker St, Lakeland, FL 33815",
    "PO Box 123, Lakeland, FL 33801",
]

batch_df = locate_houses_batch_ephemeral(SAMPLE_US_ADDRESSES)
batch_df[[
    "query_address", "house_zone", "house_score", "house_status",
    "door_zone", "door_status", "door_score", "door_rms_m",
    "ocr_status", "verified_ocr", "house_lat", "house_lng",
    "door_lat", "door_lng", "persistable_house", "persistable_door",
]]

## 9. Summary & Compliant Production Batch Architecture

* **Decoupled 2-Use-Case Architecture (House vs. Conditional Front-Door)**:
  * **Use Case 1 — Address $\rightarrow$ House Detection**: Achieves **81.0% `GREEN`** (`93.8%` main-building precision, `0.00 m` median wall distance) on `N = 100` US addresses (`balanced`) because visible houses behind trees or parked cars are no longer downgraded when their front door is hidden.
  * **Use Case 2 — Conditional Front-Door Localization**: Given a detected house, pure `door`-ray Cauchy RANSAC achieves **55.9% `GREEN`** (`94.2%` precision, `13 cm` median ray RMS, `1.66 m` pure-SVI held-out entrance median error / `1.12 m` ephemeral UI wall-clamp) plus **22.6% `YELLOW`** (`2-ray` / `PORCH_FALLBACK`) and cleanly reports **21.5% `OCCLUDED`** (`door_lat=None`) without false door pins.
* **Fast `gemini-4-hydrogen` (`LOW`, `1280 px`) + Float-BBox `UNNEST(@batch_pts)`**: Cuts interactive end-to-end latency to **~10–15 s/address** (`PIL.Image.draft` 1/2-DCT decode + Float-BBox predicate pushdown before `ST_DWITHIN`), while keeping `0` persistent GMP tables in BigQuery.
* **GA Geocoding v4 + SVI-Track Orientation (`0` Routes Pro SKU)**: Gates non-physical (`PO Box`/`RR`/`HC`), unresolved explicit-unit (`UNRESOLVED_UNIT_DESTINATION`), and campus (`GROUNDS` / `> 1,500 m²`) addresses before imagery spend and snaps inside-polygon or missing street references to the nearest `SV_PANO` car trajectory.

### Production Storage & Governance Checklist
1. **What you may persist**: Store `place_id` indefinitely, geocoded `(lat, lng)` for `<= 30` consecutive calendar days, and pure SVI/LiDAR Customer Detections (`house_lat/lng` when `persistable_house=True`, `door_lat/lng` when `persistable_door=True`, plus permitted Logistics access attributes), enforcing SVI's **35-day snapshot TTL** on `observation_id` metadata.
2. **What must stay ephemeral (`0-day` cache)**: Never persist `displayPolygon`, `navigationPoints`, `entrances`, `subDestinations`, `formattedAddress`, `building_centroid_ephemeral_lat/lng`, `door_display_ephemeral_lat/lng`, or raw `gcs_uri` images; never overlay GMP geometry on non-Google basemaps; and never use GMP Geocoding outputs as ML training/evaluation labels (`GMP ToS §3.2.3(c)(vii)`).
3. **Cost optimization**: Submit zoom crops via Vertex AI Batch Prediction (`50%` lower token rate on `gemini-4-hydrogen`: `$0.75/M` input, `$3.75/M` output, `~$0.032`/address) and ensure your public privacy policy includes the **Analytics SST §3.5.1 Residential Delivery Privacy Notice**.